# Warehouse AGV 5 Robot A*

Notebook hợp nhất từ `NHÓM_1_SMART_LOG.ipynb`, `Warehouse_AGV_3D.ipynb` và các ý tưởng điều phối trong `Draft_simul.ipynb`.

Pipeline chính: dữ liệu Kaggle → FEFO → bố trí kho 3D/WMS → A* trên lưới 0,5 m → điều phối 5 AGV → CPA, điều tốc, swept-gap, replan/detour/scan → PICK → DROP → RETURN → DOCK_TURN → CHARGE → lịch sử và CSV.

Mục tiêu của bản này là giữ đầy đủ chức năng kho và dùng một bộ điều khiển đội 5 robot duy nhất. Robot không đi xuyên kệ, không vào vùng an toàn của robot khác, và chỉ di chuyển khi kiểm tra bước quét vẫn bảo đảm `SAFE_GAP`.


In [ ]:
%pip -q install kagglehub


In [ ]:
# ==============================================================================
# MÔ HÌNH 3D KHO HÀNG FEFO - GOOGLE COLAB (NÂNG CẤP ĐỒ HỌA CHÂN THỰC 3D)
# BỔ SUNG NÂNG CẤP VISUAL:
# 1. Sàn Epoxy/Gạch Men bóng với BumpMap gờ ron, vạch chỉ dẫn giao thông & vết bánh xe
# 2. Tường Sandwich Panel công nghiệp, gờ chân tường 3D, biển báo an toàn dán tường
# 3. Đường ống dẫn kỹ thuật 3D chạy dọc tường & trần
# 4. Tay nhân viên FPS cầm PDA Scanner, Cửa cuốn tự động & Cửa nhôm kính
# 5. Đội 5 robot quản lý Pin, Đếm bước, Thời gian, Trạm sạc 3D & Mini-Map
# ==============================================================================

import os
import glob
import json
import base64
import numpy as np
import pandas as pd
from IPython.display import HTML, display

print("⏳ Đang tải dữ liệu Kaggle 'willianoliveiragibin/grocery-inventory'...")

# ------------------------------------------------------------------------------
# 1. TẢI DATASET KAGGLE
# ------------------------------------------------------------------------------
# CHỈ lấy dữ liệu gốc từ Kaggle. Không fallback sang dữ liệu bịa/mẫu.
try:
    import kagglehub
except ImportError as exc:
    raise RuntimeError("Chưa có kagglehub. Trên Colab chạy: %pip install kagglehub") from exc

try:
    dataset_path = kagglehub.dataset_download("willianoliveiragibin/grocery-inventory")
except Exception as exc:
    raise RuntimeError("Không tải được dataset Kaggle. Kiểm tra Internet/quyền truy cập rồi chạy lại.") from exc

csv_files = sorted(glob.glob(os.path.join(dataset_path, "**", "*.csv"), recursive=True))
if not csv_files:
    raise RuntimeError("Dataset tải về không có tệp CSV; không được thay bằng dữ liệu giả.")

# Ưu tiên file có đủ tên trường inventory; kiểm tra thay vì tự chế giá trị.
df_raw = None
for csv_file in csv_files:
    candidate = pd.read_csv(csv_file)
    candidate.columns = [str(c).strip().replace(" ", "_") for c in candidate.columns]
    cols = set(candidate.columns)
    if {"Product_ID", "Product_Name", "Expiration_Date", "Stock_Quantity"}.issubset(cols):
        df_raw = candidate
        print(f"✅ Dataset gốc: {os.path.basename(csv_file)} | {len(df_raw)} bản ghi")
        break
if df_raw is None:
    raise ValueError("CSV không có đủ các cột gốc: Product_ID, Product_Name, Expiration_Date, Stock_Quantity. "
                     + "Các tệp tìm thấy: " + ", ".join(os.path.basename(f) for f in csv_files))

today = None  # Chọn tự động theo ngày hết hạn của chính dataset, không sửa nguồn.
inventory_list = []
from collections import Counter
reject_counts = Counter()
parsed_dates = []

# Dữ liệu nguồn có nhiều biểu diễn ngày (M/D/YYYY, MM-DD-YYYY, ISO...)
# Chuyển đổi riêng từng giá trị để không bị suy diễn format từ dòng đầu.
def read_dataset_date(value):
    if pd.isna(value):
        return pd.NaT
    if isinstance(value, pd.Timestamp):
        result = value
    else:
        raw = str(value).strip()
        if not raw:
            return pd.NaT
        # Source uses US-style dates in mixed separators; date-first would change facts.
        try:
            result = pd.to_datetime(raw, format="mixed", dayfirst=False, errors="coerce")
        except (ValueError, TypeError):
            result = pd.to_datetime(raw, dayfirst=False, errors="coerce")
    if pd.isna(result):
        return pd.NaT
    if getattr(result, "tzinfo", None) is not None:
        result = result.tz_localize(None)
    return result.normalize()

# Chọn ngày mô phỏng dựa trên ngày hết hạn THẬT trong dataset.
# Ưu tiên ngày có đủ cả 3 nhóm FEFO và nhiều sản phẩm hợp lệ.
valid_expiries = []
for _, row in df_raw.iterrows():
    if pd.isna(row.get("Product_ID")) or pd.isna(row.get("Product_Name")):
        continue
    expiry = read_dataset_date(row.get("Expiration_Date"))
    qty = pd.to_numeric(row.get("Stock_Quantity"), errors="coerce")
    if pd.notna(expiry) and pd.notna(qty) and qty > 0:
        valid_expiries.append(expiry)
if not valid_expiries:
    raise ValueError("Dataset không có sản phẩm với ngày hết hạn đọc được và tồn kho > 0. Không tự tạo dữ liệu giả.")

import numpy as np
expiry_days = np.sort(np.array([d.value // 86_400_000_000_000 for d in valid_expiries], dtype=np.int64))
# Các ranh giới 0–15, 16–45, >45 ngày chỉ thay đổi khi qua một ngày hết hạn.
# Thử các ngày liền kề những ranh giới để tìm ngày cân đối nhất.
candidates = set()
for d in np.unique(expiry_days):
    for offset in (0, 1, 15, 16, 45, 46):
        candidates.add(int(d - offset))

best_score = None
best_day = None
best_counts = None
for day in sorted(candidates):
    red_end = np.searchsorted(expiry_days, day + 15, side="right")
    expired_end = np.searchsorted(expiry_days, day, side="left")
    yellow_end = np.searchsorted(expiry_days, day + 45, side="right")
    red = int(red_end - expired_end)
    yellow = int(yellow_end - red_end)
    green = int(len(expiry_days) - yellow_end)
    active = red + yellow + green
    nonempty = int(red > 0) + int(yellow > 0) + int(green > 0)
    # First achieve 3 groups, then optimize the smallest group,
    # then maximize variety of usable records, then prefer a later date.
    score = (nonempty, min(red, yellow, green), active, day)
    if best_score is None or score > best_score:
        best_score, best_day, best_counts = score, day, (red, yellow, green)

today = pd.Timestamp("1970-01-01") + pd.Timedelta(days=best_day)
print(f"📅 Ngày vận hành mô phỏng được tự chọn: {today:%d/%m/%Y}")
print(f"📦 Nhóm dự kiến từ dữ liệu thật: 🔴 {best_counts[0]} gấp, 🟡 {best_counts[1]} trung bình, 🟢 {best_counts[2]} không gấp")
if best_score[0] < 3:
    print("⚠️ Dataset không đủ sản phẩm để có cả ba nhóm FEFO tại cùng một ngày; không tạo dữ liệu giả.")

for _, row in df_raw.iterrows():
    product_id = row.get("Product_ID")
    name = row.get("Product_Name")
    expiry = read_dataset_date(row.get("Expiration_Date"))
    qty = pd.to_numeric(row.get("Stock_Quantity"), errors="coerce")
    if pd.notna(expiry):
        parsed_dates.append(expiry)
    if pd.isna(product_id) or pd.isna(name):
        reject_counts["thiếu tên/mã sản phẩm"] += 1
        continue
    if pd.isna(expiry):
        reject_counts["không đọc được ngày hết hạn"] += 1
        continue
    if pd.isna(qty):
        reject_counts["thiếu/sai số lượng tồn kho"] += 1
        continue
    if qty <= 0:
        reject_counts["tồn kho <= 0"] += 1
        continue
    days_left = int((expiry - today).days)
    if days_left < 0:
        reject_counts["đã hết hạn trước ngày mô phỏng"] += 1
        continue
    sku = row.get("SKU") if "SKU" in df_raw.columns else product_id
    if pd.isna(sku):
        sku = product_id
    category = row.get("Category") if "Category" in df_raw.columns else row.get("Catagory", None)
    inventory_list.append({
        "productId": str(product_id),
        "sku": str(sku),
        "productName": str(name),
        "category": str(category) if pd.notna(category) else "Không có trong dữ liệu",
        "quantity": int(qty),
        "daysLeft": days_left,
        "expiryDateStr": expiry.strftime("%d/%m/%Y")
    })

print("📋 KẾT QUẢ ĐỌC & LỌC DATASET THẬT")
print(f"  Số dòng gốc: {len(df_raw)} | Hợp lệ ở ngày {today:%d/%m/%Y}: {len(inventory_list)}")
if parsed_dates:
    print(f"  Khoảng ngày hết hạn đọc được: {min(parsed_dates):%d/%m/%Y} — {max(parsed_dates):%d/%m/%Y}")
for reason, count in reject_counts.items():
    print(f"  ⚠️ Loại {count} dòng: {reason}")
if not inventory_list:
    print("  🔎 Ví dụ 5 giá trị Expiration_Date nguyên bản:", df_raw["Expiration_Date"].head(5).tolist())
    raise ValueError(
        "Không còn sản phẩm nào hợp lệ tại ngày vận hành được chọn. Hãy xem thống kê phía trên: "
        "nếu ngày hết hạn đều trước mốc này thì không thể tạo hàng chưa hết hạn "
        "từ dữ liệu thật. Chương trình KHÔNG tự đổi ngày vận hành hay bịa sản phẩm."
    )

inventory_list.sort(key=lambda item: item["daysLeft"])
# Each inventory record is a real Kaggle row; FEFO groups use the auto-selected operation date.
from collections import Counter
group_count = Counter("GẤP" if r["daysLeft"] <= 15 else
                      "TRUNG BÌNH" if r["daysLeft"] <= 45 else "KHÔNG GẤP"
                      for r in inventory_list)
print("📊 Dữ liệu nguồn hợp lệ theo FEFO (trước khi phân bổ lên kệ):")
for group in ("GẤP", "TRUNG BÌNH", "KHÔNG GẤP"):
    print(f"   {group}: {group_count[group]} sản phẩm")
missing_groups = [g for g in ("GẤP", "TRUNG BÌNH", "KHÔNG GẤP") if not group_count[g]]
if missing_groups:
    print("⚠️ Dataset KHÔNG có hàng phù hợp cho nhóm: " + ", ".join(missing_groups)
          + ". Sẽ để thiếu nhóm này, KHÔNG tạo dữ liệu giả.")
print(f"📦 Sử dụng {len(inventory_list)} bản ghi thật từ Kaggle. Ngày vận hành: {today.strftime('%d/%m/%Y')}.")
# Vị trí sắp xếp trên các kệ 3D là bố trí MÔ PHỎNG; không phải vị trí kho thật từ dataset.
inventory_json_str = json.dumps(inventory_list, ensure_ascii=False)

# ------------------------------------------------------------------------------
# 2. TEMPLATE HTML / THREE.JS VỚI SÀN BUMP MAP & TƯỜNG PANELS SIÊU THỰC
# ------------------------------------------------------------------------------
html_template = """<!DOCTYPE html>
<html lang="vi">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Mô Phỏng Kho Hàng 3D FEFO - Photorealistic Environment</title>
    <!-- Tailwind CSS & FontAwesome -->
    <script async src="https://cdn.tailwindcss.com"></script>
    <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css">
    <!-- Three.js Engine & Controls -->
    <script async src="https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js"></script>
    <script async src="https://cdn.jsdelivr.net/npm/three@0.128.0/examples/js/controls/OrbitControls.js"></script>
    <style>
        body { margin: 0; overflow: hidden; font-family: 'Segoe UI', system-ui, sans-serif; background-color: #090d16; color: white; user-select: none; }
        #canvas-container { width: 100vw; height: 100vh; position: relative; overflow: hidden; }
        #canvas-container canvas { display:block; }
        #canvas-container:fullscreen, #canvas-container:-webkit-full-screen { width:100vw; height:100vh; }

        #render-error-panel {
            position:fixed;left:50%;top:14%;transform:translateX(-50%);z-index:100000;
            width:min(660px,90vw);max-height:65vh;overflow:auto;
            padding:20px 24px;border:2px solid #ef4444;border-radius:16px;
            background:rgba(15,23,42,.98);color:#fff;
            box-shadow:0 16px 60px rgba(0,0,0,.65);font-size:14px;
            display:none;white-space:pre-wrap;user-select:text;
        }
        .crosshair {
            position: absolute; top: 50%; left: 50%;
            width: 10px; height: 10px;
            transform: translate(-50%, -50%);
            pointer-events: none; z-index: 20;
        }
        .crosshair::before, .crosshair::after {
            content: ''; position: absolute; background: #ef4444; box-shadow: 0 0 6px #ef4444;
        }
        .crosshair::before { top: 4px; left: -3px; width: 16px; height: 2px; }
        .crosshair::after { top: -3px; left: 4px; width: 2px; height: 16px; }

        .glass-panel {
            background: rgba(255, 255, 255, 0.93);
            backdrop-filter: blur(12px);
            border: 2px solid #dc2626;
            box-shadow: 0 10px 25px rgba(220, 38, 38, 0.25);
            color: #1e293b;
        }

        .tablet-chassis {
            background: #18181b;
            border: 14px solid #27272a;
            border-radius: 36px;
            box-shadow: 0 30px 60px rgba(0,0,0,0.9), 0 0 0 2px #52525b, inset 0 0 12px rgba(255,255,255,0.1);
            position: relative;
        }
        .tablet-camera {
            position: absolute; top: -8px; left: 50%; transform: translateX(-50%);
            width: 10px; height: 10px; background: #09090b; border-radius: 50%;
            border: 2px solid #3f3f46; z-index: 50;
        }
        .tablet-screen {
            background: linear-gradient(135deg, #f8fafc 0%, #f1f5f9 100%);
            border-radius: 22px;
            color: #0f172a;
            overflow: hidden;
        }
        .tab-btn.active {
            background: #dc2626; color: white; font-weight: bold;
            box-shadow: 0 4px 12px rgba(220, 38, 38, 0.4);
        }
        .hud-left-card { max-width: min(58vw, 760px); min-width: 0; }
        .hud-title-wrap { min-width: 0; }
        .hud-status-pill { display:inline-block; max-width:min(26vw, 340px); white-space:nowrap; overflow:hidden; text-overflow:ellipsis; vertical-align:middle; }
        #route-result-box { align-items:flex-start; flex-wrap:wrap; gap:16px; }
        #route-result-box > div { min-width:0; }
        .tablet-screen .tab-content { min-height:0; }
    </style>
</head>
<body>
    <div id="canvas-container">
        <div id="render-error-panel" role="alert"></div>
        <!-- Tâm ngắm -->
        <div id="crosshair" class="crosshair"></div>

        <!-- HUD Header Bar -->
        <div class="absolute top-3 left-3 right-3 flex justify-between items-center gap-2 z-10 pointer-events-none">
            <div class="glass-panel hud-left-card px-4 py-2 rounded-2xl flex items-center gap-3 pointer-events-auto">
                <div class="w-3.5 h-3.5 rounded-full bg-red-600 animate-ping"></div>
                <div class="hud-title-wrap">
                    <h1 class="text-sm font-black text-slate-900 tracking-wide flex items-center gap-2 min-w-0">
                        <span>KHO FEFO PHOTOREALISTIC · __OPERATION_DATE__</span>
                        <span class="text-[10px] bg-red-100 text-red-600 px-2 py-0.5 rounded-full font-bold border border-red-300 hud-status-pill" id="agv-hud-status" title="🤖 ĐỘI ROBOT: SẴN SÀNG">🤖 ĐỘI ROBOT: SẴN SÀNG</span>
                    </h1>
                    <p class="text-[11px] text-slate-600 font-semibold">
                        ⚡ Pin: <b class="text-red-600" id="agv-hud-battery">100%</b> •
                        🚶‍♂️ Bước: <b class="text-slate-900 font-mono" id="agv-hud-steps">0</b> •
                        ⏱️ Thời gian: <b class="text-red-600 font-mono" id="agv-hud-time">0.0s</b>
                    </p>
                </div>
            </div>

            <!-- Toolbar Header -->
            <div class="flex gap-2 pointer-events-auto flex-wrap justify-end shrink-0">
                <button onclick="toggleFullscreen()" class="glass-panel hover:bg-red-50 text-red-600 text-xs font-bold px-3 py-2 rounded-2xl flex items-center gap-2 transition">
                    <i class="fa-solid fa-expand text-red-600"></i>
                    <span>Toàn Màn Hình</span>
                </button>
                <button onclick="toggleTablet()" class="bg-red-600 hover:bg-red-700 text-white font-bold text-xs px-4 py-2 rounded-2xl flex items-center gap-2 shadow-lg shadow-red-600/30 transition transform active:scale-95">
                    <i class="fa-solid fa-tablet-screen-button text-base"></i>
                    <span>TABLET WMS</span>
                </button>
                <button id="btn-mode" onclick="toggleViewMode()" class="glass-panel hover:bg-slate-100 text-slate-800 text-xs font-bold px-4 py-2 rounded-2xl flex items-center gap-2 transition">
                    <i class="fa-solid fa-person-walking text-red-600"></i>
                    <span id="mode-text">Góc Nhìn Nhân Viên (FPS)</span>
                </button>
            </div>
        </div>

        <!-- MÀN HÌNH MÁY TÍNH BẢNG TABLET WMS -->
        <div id="tablet-modal" class="absolute inset-0 z-40 flex items-center justify-center bg-black/75 backdrop-blur-md hidden">
            <div class="tablet-chassis flex flex-col p-3" style="width:min(1380px,calc(100vw - 28px));height:min(920px,calc(100vh - 28px));max-width:100%;max-height:100%">
                <div class="tablet-camera"></div>

                <div class="tablet-screen flex-1 min-h-0 flex flex-col p-4 text-xs relative">

                    <!-- Tablet Status Bar -->
                    <div class="flex justify-between items-center text-[11px] text-slate-500 font-bold border-b border-slate-200 pb-2 mb-3">
                        <div class="flex items-center gap-2 text-slate-700">
                            <i class="fa-solid fa-warehouse text-red-600 text-sm"></i>
                            <span>WMS OS v5.0 • FEFO SIMULATION</span>
                        </div>
                        <div class="flex items-center gap-3">
                            <span class="text-red-600">📅 Ngày vận hành: __OPERATION_DATE__</span> <span id="tablet-clock">12:00 PM</span>
                            <i class="fa-solid fa-wifi text-slate-700"></i>
                            <span class="text-slate-700"><i class="fa-solid fa-battery-full text-emerald-600"></i> 98%</span>
                        </div>
                    </div>

                    <!-- Tablet Tabs -->
                    <div class="flex gap-2 mb-3 flex-wrap">
                        <button onclick="switchTab('tab-robot')" id="btn-tab-robot" class="tab-btn active px-4 py-2 rounded-xl text-slate-700 transition flex items-center gap-2 bg-slate-200">
                            <i class="fa-solid fa-robot"></i> PHÁT LỆNH ROBOT
                        </button>
                        <button onclick="switchTab('tab-search')" id="btn-tab-search" class="tab-btn px-4 py-2 rounded-xl text-slate-700 transition flex items-center gap-2 bg-slate-200">
                            <i class="fa-solid fa-boxes-stacked"></i> TRA CỨU SẢN PHẨM & SKU
                        </button>
                        <button onclick="switchTab('tab-summary')" id="btn-tab-summary" class="tab-btn px-4 py-2 rounded-xl text-slate-700 transition flex items-center gap-2 bg-slate-200">
                            <i class="fa-solid fa-chart-pie"></i> THỐNG KÊ FEFO
                        </button>

                        <button onclick="switchTab('tab-history')" id="btn-tab-history" class="tab-btn px-4 py-2 rounded-xl text-slate-700 transition flex items-center gap-2 bg-slate-200">
                            <i class="fa-solid fa-clock-rotate-left"></i> LỊCH SỬ LẤY HÀNG
                        </button>
                        <button onclick="toggleTablet()" class="ml-auto w-8 h-8 rounded-xl bg-slate-200 hover:bg-red-500 text-slate-600 hover:text-white flex items-center justify-center transition">
                            <i class="fa-solid fa-xmark"></i>
                        </button>
                    </div>

                    <!-- TAB 1: ĐIỀU KHIỂN THỐNG NHẤT -->
                    <div id="tab-robot" class="tab-content flex-1 flex flex-col gap-3 overflow-y-auto min-h-0">
                        <div class="grid grid-cols-4 gap-3 bg-white p-2.5 rounded-2xl border border-slate-200 shadow-sm">
                            <div class="bg-slate-50 p-2 rounded-xl border border-slate-200 flex items-center gap-2.5">
                                <div class="w-8 h-8 rounded-lg bg-red-100 text-red-600 flex items-center justify-center text-sm"><i class="fa-solid fa-bolt"></i></div>
                                <div>
                                    <div class="text-[10px] text-slate-400 font-bold">PIN ROBOT CHỌN</div>
                                    <div class="font-bold text-red-600 text-xs font-mono" id="agv-tab-battery-text">100% (10 đơn)</div>
                                </div>
                            </div>
                            <div class="bg-slate-50 p-2 rounded-xl border border-slate-200 flex items-center gap-2.5">
                                <div class="w-8 h-8 rounded-lg bg-slate-200 text-slate-800 flex items-center justify-center text-sm"><i class="fa-solid fa-shoe-prints"></i></div>
                                <div>
                                    <div class="text-[10px] text-slate-400 font-bold">BƯỚC ROBOT CHỌN</div>
                                    <div class="font-bold text-slate-800 text-xs font-mono" id="agv-tab-current-steps">0 bước</div>
                                </div>
                            </div>
                            <div class="bg-slate-50 p-2 rounded-xl border border-slate-200 flex items-center gap-2.5">
                                <div class="w-8 h-8 rounded-lg bg-red-100 text-red-600 flex items-center justify-center text-sm"><i class="fa-solid fa-stopwatch"></i></div>
                                <div>
                                    <div class="text-[10px] text-slate-400 font-bold">THỜI GIAN ROBOT CHỌN</div>
                                    <div class="font-bold text-red-600 text-xs font-mono" id="agv-tab-current-time">0.0s</div>
                                </div>
                            </div>
                            <div class="bg-slate-50 p-2 rounded-xl border border-slate-200 flex items-center gap-2.5">
                                <div class="w-8 h-8 rounded-lg bg-slate-200 text-slate-800 flex items-center justify-center text-sm"><i class="fa-solid fa-chart-line"></i></div>
                                <div>
                                    <div class="text-[10px] text-slate-400 font-bold">TỔNG TÍCH LŨY</div>
                                    <div class="font-bold text-slate-800 text-[11px] font-mono" id="agv-tab-total-stats">0 bước | 00:00</div>
                                </div>
                            </div>
                        </div>

                        <div class="grid grid-cols-12 gap-3 flex-1 min-h-[245px]">
                            <div class="col-span-5 bg-white p-3 rounded-2xl border border-slate-200 shadow-sm flex flex-col gap-3">
                                <div class="font-bold text-red-600 text-xs border-b border-slate-100 pb-1.5 flex justify-between items-center">
                                    <span>📦 TẠO DANH SÁCH PHÁT LỆNH</span>
                                    <span class="text-[10px] bg-red-100 text-red-600 px-2 py-0.5 rounded-full" id="agv-tab-badge">IDLE</span>
                                </div>
                                <div>
                                    <label class="font-bold text-slate-700 block mb-1">1. Tìm sản phẩm / SKU</label>
                                    <input type="text" id="agv-sku-input" oninput="filterAGVProductList()" placeholder="Tìm theo SKU, PID, tên sản phẩm..." class="w-full bg-slate-100 border border-slate-200 text-slate-800 px-2.5 py-1.5 rounded-lg text-xs focus:outline-none focus:border-red-500 mb-1">
                                    <select id="agv-select-item" onchange="syncSelectedSlotToMap()" class="w-full bg-slate-50 border border-slate-200 text-slate-800 px-2.5 py-1.5 rounded-lg text-xs focus:outline-none focus:border-red-500"></select>
                                </div>
                                <div>
                                    <label for="fleet-select-robot" class="font-bold text-slate-700 block mb-1">2. Chọn robot thực hiện</label>
                                    <select id="fleet-select-robot" onchange="f5RefreshTable()" class="w-full bg-slate-50 border border-slate-200 text-slate-800 px-2.5 py-1.5 rounded-lg text-xs">
                                        <option value="0">Robot 1</option><option value="1">Robot 2</option><option value="2">Robot 3</option>
                                        <option value="3">Robot 4</option><option value="4">Robot 5</option>
                                    </select>
                                    <div class="text-[10px] text-slate-500 mt-1" id="agv-status-detail">Chọn robot, SKU và điểm giao để tạo dòng chờ lệnh</div>
                                </div>
                                <div>
                                    <label for="agv-select-drop" class="font-bold text-slate-700 block mb-1">3. Chọn địa điểm giao</label>
                                    <select id="agv-select-drop" onchange="syncDropLocationFromSelect()" class="w-full bg-slate-50 border border-slate-200 text-slate-800 px-2.5 py-1.5 rounded-lg text-xs">
                                        <option value="DESPATCH">🚪 Cửa xuất hàng (CuaXuat)</option>
                                        <option value="INBOUND">🚪 Cửa nhập hàng (CuaNhap)</option>
                                    </select>
                                </div>
                                <div class="bg-slate-50 rounded-xl px-3 py-2 text-[11px] text-slate-600">
                                    <div>⚡ Điều tốc tự động: <b id="fleet-speed-setting" class="text-red-600">Tối đa 2.3 m/s</b></div>
                                    <div class="text-[10px] mt-1">Tự tăng/giảm theo khoảng cách an toàn, robot lân cận và góc cua. Chỉ dừng khi phía trước bị chặn.</div>
                                    <div class="text-[10px]">ETA dựa trên đường đi và vận tốc tối đa; thời gian thực tế tính trong lúc chạy.</div>
                                </div>
                                <div class="grid grid-cols-2 gap-2">
                                    <button onclick="addPendingDispatch()" class="w-full bg-slate-800 hover:bg-slate-900 text-white font-black py-2.5 rounded-xl shadow-md">
                                        <i class="fa-solid fa-list-check"></i> XÁC NHẬN CHỌN
                                    </button>
                                    <button onclick="dispatchPendingQueue()" class="w-full bg-red-600 hover:bg-red-700 text-white font-black py-2.5 rounded-xl shadow-md shadow-red-600/20">
                                        <i class="fa-solid fa-paper-plane"></i> PHÁT LỆNH
                                    </button>
                                </div>
                                <button onclick="clearPendingDispatches()" class="w-full bg-slate-200 hover:bg-slate-300 text-slate-700 font-bold py-2 rounded-xl">
                                    <i class="fa-solid fa-trash"></i> XÓA DANH SÁCH CHỜ
                                </button>
                                <div class="bg-slate-50 border border-slate-200 rounded-xl p-2.5 flex-1 min-h-[145px] flex flex-col">
                                    <div class="flex items-center justify-between mb-2">
                                        <div class="font-bold text-slate-700">📝 DÒNG CHỜ PHÁT LỆNH</div>
                                        <div id="pending-count-badge" class="text-[10px] bg-red-100 text-red-600 px-2 py-0.5 rounded-full">0 lệnh</div>
                                    </div>
                                    <div class="overflow-auto flex-1">
                                        <table class="w-full text-[11px]">
                                            <thead class="text-slate-500 sticky top-0 bg-slate-50">
                                                <tr><th class="text-left py-1">SKU</th><th class="text-left py-1">Robot</th><th class="text-left py-1">Điểm giao</th><th class="text-right py-1">Xóa</th></tr>
                                            </thead>
                                            <tbody id="pending-dispatch-body"></tbody>
                                        </table>
                                    </div>
                                </div>
                                <div id="dispatch-feedback" role="status" class="text-[11px] text-slate-600">Chọn SKU, robot và điểm giao rồi Xác nhận chọn. Có thể chọn lại cùng robot cho các đơn tiếp theo. Nhấn Phát lệnh một lần: mỗi robot xử lý tuần tự các đơn của mình rồi mới về trạm sạc.</div>
                            </div>

                            <div class="col-span-7 bg-white p-3 rounded-2xl border border-slate-200 shadow-sm flex flex-col justify-between relative">
                                <div class="flex justify-between items-center mb-1">
                                    <span class="font-bold text-red-600 text-xs"><i class="fa-solid fa-map"></i> SƠ ĐỒ KHO 2D MINI-MAP</span>
                                    <span class="text-[10px] text-slate-400">⚡ Sạc | ①–⑤ Robot | Đường màu: lộ trình</span>
                                </div>
                                <div class="flex-1 bg-slate-900 rounded-xl border border-slate-200 relative flex items-center justify-center overflow-hidden">
                                    <canvas id="minimap-canvas" width="460" height="230" class="cursor-pointer"></canvas>
                                </div>
                                <div class="text-[10px] text-slate-500 mt-1 flex justify-between font-semibold">
                                    <span>Vị trí chọn: <b class="text-red-600 font-mono" id="minimap-selected-loc">CuaXuat</b></span>
                                    <span>Robot 1–5 cùng thực hiện lệnh theo mẫu chung</span>
                                </div>
                            </div>
                         </div>

                     </div>

                     <!-- TAB 2: TRA CỨU -->
                    <div id="tab-search" class="tab-content flex-1 flex-col gap-3 hidden">
                        <div class="grid grid-cols-3 gap-3">
                            <div class="col-span-2 bg-white p-3 rounded-2xl border border-slate-200 shadow-sm flex flex-col gap-2">
                                <label class="font-bold text-slate-800">🔍 Tìm theo SKU / Product_ID / Tên SP:</label>
                                <div class="flex gap-2">
                                    <input type="text" id="tablet-search-input" placeholder="Nhập SKU (vd: SKU-20001)..." class="flex-1 bg-slate-100 border border-slate-200 rounded-xl px-3 py-2 text-slate-800 focus:outline-none focus:border-red-500">
                                    <button onclick="searchAndNavigate()" class="bg-red-600 hover:bg-red-700 text-white font-bold px-4 py-2 rounded-xl transition flex items-center gap-1">
                                        <i class="fa-solid fa-location-arrow"></i> Dẫn Đường 3D
                                    </button>
                                </div>
                            </div>
                            <div class="bg-white p-3 rounded-2xl border border-slate-200 shadow-sm">
                                <div class="font-bold text-red-600 mb-1">⚡ Ưu tiên FEFO:</div>
                                <button onclick="navigateCriticalFEFO()" class="w-full bg-red-100 hover:bg-red-200 text-red-700 font-bold py-2 px-3 rounded-xl transition flex items-center justify-between mt-1">
                                    <span>🔴 Đến SP Cận Hạn Nhất</span>
                                    <i class="fa-solid fa-angle-right"></i>
                                </button>
                            </div>
                        </div>
                        <div id="route-result-box" class="flex-1 bg-white rounded-2xl p-4 border border-slate-200 shadow-sm flex justify-between items-start hidden overflow-auto">
                            <div class="space-y-1">
                                <span class="bg-emerald-100 text-emerald-700 px-2 py-0.5 rounded font-bold uppercase text-[10px]" id="tab-res-status">🟢 AN TOÀN</span>
                                <h3 class="text-base font-bold text-slate-900 mt-1" id="tab-res-title">Sữa tươi Vinamilk 1L</h3>
                                <div class="text-slate-600 grid grid-cols-2 gap-x-6 gap-y-1 text-xs pt-1">
                                    <div>• SKU: <b class="text-red-600 font-mono" id="tab-res-sku">SKU-20022</b></div>
                                    <div>• Mã Kệ: <b class="text-slate-900 font-mono" id="tab-res-loc">H02T04</b></div>
                                    <div>• Hạn dùng còn: <b class="text-red-600" id="tab-res-days">42 Ngày</b></div>
                                    <div>• Tồn kho: <b class="text-slate-900" id="tab-res-qty">120 đơn vị</b></div>
                                </div>
                            </div>
                            <div class="text-right border-l border-slate-200 pl-6 space-y-1 shrink-0">
                                <div class="text-slate-400 text-[10px]">ĐỘ DÀI QUÃNG ĐƯỜNG</div>
                                <div class="text-2xl font-black text-red-600" id="tab-res-dist">18.4 m</div>
                            </div>
                        </div>
                    </div>

                    <!-- TAB 3: SUMMARY -->
                    <div id="tab-summary" class="tab-content flex-1 flex-col gap-3 hidden">
                        <div id="fefo-stock-diversity" class="bg-white border border-slate-200 rounded-lg px-3 py-2 text-slate-700 font-bold">FEFO: 🔴 Đỏ ≤15 ngày · 🟡 Vàng 16–45 ngày · 🟢 Xanh &gt;45 ngày</div>
                        <div class="grid grid-cols-3 gap-3">
                            <div class="bg-red-50 border border-red-200 p-3 rounded-2xl text-center">
                                <div class="text-red-600 text-xs font-bold">🔴 CẦN XUẤT GẤP</div>
                                <div class="text-2xl font-black text-slate-900 my-1" id="tab-sum-red">0</div>
                                <div class="text-[10px] text-slate-500">&lt; 15 Ngày (Gần cửa)</div>
                            </div>
                            <div class="bg-amber-50 border border-amber-200 p-3 rounded-2xl text-center">
                                <div class="text-amber-600 text-xs font-bold">🟡 VÀNG - SẮP ĐẾN HẠN</div>
                                <div class="text-2xl font-black text-slate-900 my-1" id="tab-sum-orange">0</div>
                                <div class="text-[10px] text-slate-500">16 - 45 Ngày</div>
                            </div>
                            <div class="bg-emerald-50 border border-emerald-200 p-3 rounded-2xl text-center">
                                <div class="text-emerald-600 text-xs font-bold">🟢 XANH - HẠN CÒN DÀI</div>
                                <div class="text-2xl font-black text-slate-900 my-1" id="tab-sum-green">0</div>
                                <div class="text-[10px] text-slate-500">&gt; 45 Ngày</div>
                            </div>
                        </div>
                    </div>

                    <!-- TAB 4: NHẬT KÝ LẤY HÀNG -->
                    <div id="tab-history" class="tab-content flex-1 flex-col gap-3 overflow-y-auto min-h-0 hidden">
                        <div class="bg-white border border-slate-200 rounded-2xl p-4 flex flex-wrap gap-3 items-center justify-between">
                            <div>
                                <div class="font-black text-slate-800 text-base">📋 LỊCH SỬ LẤY & GIAO HÀNG CỦA ROBOT</div>
                                <div class="text-slate-500 mt-1">Thời gian thực tế tính theo đồng hồ mô phỏng. Xung đột = số lần robot phải dừng/nhường vì robot khác; va chạm vật lý được phòng tránh.</div>
                            </div>
                            <div class="flex gap-2 items-center">
                                <b id="history-count" class="text-red-600">0 đơn hoàn thành</b>
                                <button onclick="downloadTaskHistoryCSV()" class="bg-slate-800 text-white font-bold px-3 py-2 rounded-lg">Xuất CSV</button>
                            </div>
                        </div>
                        <div class="flex-1 overflow-auto rounded-2xl border border-slate-200 bg-white">
                            <table class="w-full text-xs" style="border-collapse:collapse; min-width:920px">
                                <thead class="sticky top-0 bg-slate-100 text-slate-700"><tr>
                                    <th class="p-3 text-left">STT</th>
                                    <th class="p-3 text-left">Robot</th>
                                    <th class="p-3 text-left">SKU</th>
                                    <th class="p-3 text-right">Thời gian dự tính</th>
                                    <th class="p-3 text-right">Thời gian thực tế</th>
                                    <th class="p-3 text-right">Đã đi (m)</th>
                                    <th class="p-3 text-right">Xung đột tránh được</th>
                                    <th class="p-3 text-right">Va chạm</th>
                                </tr></thead>
                                <tbody id="task-history-body"><tr><td colspan="8" class="text-center text-slate-500 p-5">Chưa có lượt giao hàng nào hoàn tất.</td></tr></tbody>
                            </table>
                        </div>
                        <div class="text-[11px] text-slate-500">Dữ liệu lưu trong phiên chạy hiện tại. Xuất CSV để giữ lịch sử sau khi tải lại.</div>
                    </div>

                </div>
            </div>
        </div>

        <!-- Overlay Hướng dẫn -->
        <div id="instructions-overlay" class="absolute inset-0 z-30 flex flex-col justify-center items-center text-center p-6 cursor-pointer">
            <div class="glass-panel p-8 rounded-3xl max-w-md border-2 border-red-600 shadow-2xl">
                <div class="w-16 h-16 bg-red-100 text-red-600 rounded-2xl flex items-center justify-center mx-auto mb-4 text-3xl border border-red-300">
                    <i class="fa-solid fa-warehouse"></i>
                </div>
                <h2 class="text-xl font-black mb-2 text-slate-900">KHO HÀNG FEFO PHOTOREALISTIC 3D</h2>
                <p class="text-xs text-slate-600 mb-6">Sàn Gạch Men Glossy • BumpMap Ron Gạch • Tường Sandwich Panel • Ống Dẫn 3D.</p>

                <div class="grid grid-cols-2 gap-3 text-left text-xs mb-6 bg-slate-100 p-4 rounded-2xl border border-slate-200 text-slate-700">
                    <div class="flex items-center gap-2"><kbd class="px-2 py-1 bg-white rounded shadow text-red-600 font-mono">W S A D</kbd> <span>Di chuyển</span></div>
                    <div class="flex items-center gap-2"><kbd class="px-2 py-1 bg-white rounded shadow text-slate-900 font-mono">ALT</kbd> <span>Hiện con trỏ</span></div>
                    <div class="flex items-center gap-2"><kbd class="px-2 py-1 bg-white rounded shadow text-red-600 font-mono">Chuột</kbd> <span>Xoay camera</span></div>
                    <div class="flex items-center gap-2"><kbd class="px-2 py-1 bg-white rounded shadow text-red-600 font-mono">Click</kbd> <span>Kiểm tra SP</span></div>
                </div>

                <button class="w-full bg-red-600 hover:bg-red-700 text-white font-black py-3 rounded-xl transition shadow-lg shadow-red-600/30">
                    NHẤP ĐỂ BẮT ĐẦU VÀO KHO
                </button>
            </div>
        </div>

        <!-- Inspector Card -->
        <div id="inspector-panel" class="absolute bottom-4 left-4 z-20 glass-panel p-4 rounded-2xl w-80 shadow-2xl hidden border-l-8 border-l-red-600">
            <div class="flex justify-between items-start mb-2">
                <div>
                    <span id="inspect-tag" class="text-[10px] font-bold px-2 py-0.5 rounded uppercase bg-red-100 text-red-600 border border-red-200">CẦN XUẤT GẤP</span>
                    <h3 id="inspect-title" class="text-base font-bold text-slate-900 mt-1">Sữa tươi Vinamilk 1L</h3>
                </div>
                <button onclick="closeInspector()" class="text-slate-400 hover:text-slate-800 p-1"><i class="fa-solid fa-xmark"></i></button>
            </div>
            <div class="space-y-1.5 text-xs text-slate-700">
                <div class="flex justify-between py-1 border-b border-slate-200"><span class="text-slate-500">Mã Location:</span><span id="inspect-loc" class="font-mono font-bold text-red-600">--</span></div>
                <div class="flex justify-between py-1 border-b border-slate-200"><span class="text-slate-500">Mã SKU / PID:</span><span id="inspect-sku" class="font-mono font-bold text-slate-900">SKU-20042</span></div>
                <div class="flex justify-between py-1 border-b border-slate-200"><span class="text-slate-500">Vị trí Dãy Kệ:</span><span id="inspect-pos" class="font-bold text-slate-800">Main Row A - Tầng 02</span></div>
                <div class="flex justify-between py-1 border-b border-slate-200"><span class="text-slate-500">Số lượng:</span><span id="inspect-qty" class="font-bold text-emerald-600">145 đơn vị</span></div>
                <div class="flex justify-between py-1 border-b border-slate-200"><span class="text-slate-500">Hạn sử dụng:</span><span id="inspect-exp" class="font-bold text-red-600">12/11/2026 (12 ngày)</span></div>
            </div>

            <div class="grid grid-cols-2 gap-2 mt-3 pt-2 border-t border-slate-200">
                <button onclick="dispatchInspectedFleetItem()" class="bg-red-600 hover:bg-red-700 text-white py-1.5 px-2 rounded-xl text-[11px] font-bold transition flex items-center justify-center gap-1 shadow-sm">
                    <i class="fa-solid fa-robot"></i> Cho Robot Chọn Lấy
                </button>
                <button onclick="relocateInspectedItem()" class="bg-slate-200 hover:bg-slate-300 text-slate-800 py-1.5 px-2 rounded-xl text-[11px] font-bold transition flex items-center justify-center gap-1">
                    <i class="fa-solid fa-arrows-rotate"></i> Dời sang Ô Trống
                </button>
            </div>
        </div>
    </div>

    <script>
        const RAW_INVENTORY = __INVENTORY_JSON_DATA__;

        const WORLD_SIZE = 28.0;
        const GRID_RES = 0.5;
        const GRID_DIM = Math.floor(WORLD_SIZE / GRID_RES);

        const INBOUND_DOOR_POS = { x: 6.0, y: 0.15, z: 27.8 };
        const DESPATCH_DOOR_POS = { x: 22.0, y: 0.15, z: 27.8 };
        // Charging station along the LEFT wall (x≈0), parallel to the wall.
        // The inbound doorway at x=6,z=27.8 remains entirely unobstructed.
        const CHARGING_STATION_POS = { x: 1.25, y: 0.05, z: 22.35 };
        const CHARGING_BAYS = [
            {x:1.25,z:19.20}, {x:1.25,z:20.45}, {x:1.25,z:21.70},
            {x:1.25,z:22.95}, {x:1.25,z:24.20}
        ];
        const DOCK_APPROACH_X = 2.75; // stand off from the left wall: entry points along -X
        const DOCK_ENTRY_HEADING = -Math.PI/2;  // head-first toward left wall
        const DOCK_EXIT_HEADING = Math.PI/2;    // head out toward main aisle
        const DOCK_TURN_RATE = Math.PI/1.8;     // complete a 180-degree turn in 1.8 s

        let scene, camera, renderer, controls;
        let isFPSMode = true;
        let slotsData = [];
        let slotMeshes = [];
        let obstacleBoxes = [];
        let gridMap = [];

        let pathLineMesh = null;
        let highlightMesh = null;
        let currentInspectedSlot = null;

        // Player Controls & FPS Arms
        let moveForward = false, moveBackward = false, moveLeft = false, moveRight = false, isSprinting = false;
        let yaw = Math.PI, pitch = 0;
        let prevTime = performance.now();
        let isMouseDown = false;
        let prevMouseX = 0, prevMouseY = 0;

        let fpsArmGroup = null;
         let chargingStationMesh = null;



        function setHudStatus(text) {
            const el = document.getElementById('agv-hud-status');
            if (el) { el.innerText = text; el.title = text; }
        }

         // UNIFIED FLEET: exactly five robots, identical behavior and command lifecycle
         const ROBOT_COUNT=5;
         const fleet5Colors=[0xdc2626,0x3b82f6,0x22c55e,0xf59e0b,0xa855f7];
         let fleet5=[], fleet5Running=false, fleet5Clock=0;
         const taskHistory=[];
         const reservedSlotIds=new Set();
         // Orders accepted but not started must not be assigned to a second robot.
         const queuedSlotIds=new Set();
         let pendingDispatches=[];
         let fleetSpeedLog=[], fleetLogClock=0, fleetUIClock=0;
         // Autonomous cruise controller: acceleration/braking and forward/side sensing.
         const SAFE_GAP=0.98;
         const MAX_SPEED=2.3;     // absolute speed limit, m/s
         const ACCEL=1.15;        // m/s^2; gradual acceleration
         const BRAKE=2.7;        // m/s^2; safe emergency deceleration
         const SENSOR_RANGE=4.8;
         const SIDE_RANGE=1.8;
         const TURN_SPEED=0.85;
         const SLOW_SPEED=TURN_SPEED; // compatibility for unused legacy Robot 1 routines
         const ANGULAR_RATE=2.8;  // rad/s. Smooth free heading changes during driving
         const SCAN_ANGULAR_RATE=3.2; // full 360-degree stationary scan in about 2 s
         const BLOCKED_SCAN_AFTER=14.0; // only a genuine 14-second deadlock justifies a physical 360 scan
         const CONFLICT_HORIZON=2.4; // seconds of robot-motion prediction
         const REPLAN_INTERVAL=2.6;   // do not thrash/recompute an unchanged path every frame
         const SENSOR_DECISION_INTERVAL=.15; // predictive decisions held briefly for stability
         const SCAN_COOLDOWN=25.0;   // no repeated spins in a blocked corridor
         function setFleetSpeed(){ /* fixed automatic controller: no manual speed input */ }

         function f5Nearest(x,z,blocked=null) {
             const gx=Math.max(0,Math.min(GRID_DIM-1,Math.floor(x/GRID_RES)));
             const gz=Math.max(0,Math.min(GRID_DIM-1,Math.floor(z/GRID_RES)));
             for(let radius=0;radius<GRID_DIM;radius++){
                 let best=null,bestDist=Infinity;
                 for(let dx=-radius;dx<=radius;dx++)for(let dz=-radius;dz<=radius;dz++){
                     if(radius && Math.max(Math.abs(dx),Math.abs(dz))!==radius)continue;
                     const nx=gx+dx,nz=gz+dz;
                     if(nx<0||nz<0||nx>=GRID_DIM||nz>=GRID_DIM || !gridMap[nx]?.[nz]?.walkable)continue;
                     const wx=(nx+.5)*GRID_RES,wz=(nz+.5)*GRID_RES;
                     if(blocked && blocked(wx,wz))continue;
                     const dist=(wx-x)**2+(wz-z)**2;
                     if(dist<bestDist){bestDist=dist;best={x:nx,z:nz};}
                 }
                 if(best)return best;
             }
             return null;
         }
         function f5World(n){return {x:(n.x+.5)*GRID_RES,z:(n.z+.5)*GRID_RES};}
         function f5OtherPositions(exceptId){
             return fleet5.filter(r=>r.id!==exceptId).map(r=>r.mesh.position);
         }
         function f5Dist(a,b){return Math.hypot(a.x-b.x,a.z-b.z);}
         // Cooperative weighted A*: obstacles are hard, robots are SOFT costs.
         // This prevents a parked AGV from removing the only walkable corridor.
         function f5PlanCost(startWorld,goalWorld,exceptId,avoidOccupied=false,returnTrip=false){
             const start=f5Nearest(startWorld.x,startWorld.z);
             const goal=f5Nearest(goalWorld.x,goalWorld.z);
             if(!start||!goal)return [];
             const key=n=>n.x+GRID_DIM*n.z;
             const sourceKey=key(start),targetKey=key(goal);
             const gScore=new Float64Array(GRID_DIM*GRID_DIM);gScore.fill(Infinity);
             const parent=new Int32Array(GRID_DIM*GRID_DIM);parent.fill(-1);
             const closed=new Uint8Array(GRID_DIM*GRID_DIM);
             const open=[];
             // Binary heap avoids O(n log n) sorting the entire queue at each expansion.
             function push(x){let i=open.length;open.push(x);while(i){let p=(i-1)>>1;if(open[p].f<=x.f)break;open[i]=open[p];i=p;}open[i]=x;}
             function pop(){const head=open[0],last=open.pop();if(open.length){let i=0;while((i*2+1)<open.length){let child=i*2+1,right=child+1;if(right<open.length&&open[right].f<open[child].f)child=right;if(open[child].f>=last.f)break;open[i]=open[child];i=child;}open[i]=last;}return head;}
             const h=n=>Math.abs(n.x-goal.x)+Math.abs(n.z-goal.z);
             gScore[sourceKey]=0;push({n:start,f:h(start)});
             const otherRobots=avoidOccupied?fleet5.filter(r=>r.id!==exceptId):[];
             const directions=[[1,0],[-1,0],[0,1],[0,-1]];
             let found=false;
             while(open.length){
                 const {n}=pop(),k=key(n);
                 if(closed[k])continue;
                 closed[k]=1;
                 if(k===targetKey){found=true;break;}
                 for(const [dx,dz] of directions){
                     const nx=n.x+dx,nz=n.z+dz;
                     if(nx<0||nz<0||nx>=GRID_DIM||nz>=GRID_DIM||!gridMap[nx][nz].walkable)continue;
                     const v={x:nx,z:nz},nk=key(v);
                     if(closed[nk])continue;
                     const world=f5World(v);
                     // Never use the strip BEHIND charging bays as a shortcut.
                     // Only direct head-in dock motion may enter this strip.
                     // Otherwise parked robots can form an unresolvable wall.
                     if(world.x<2.22 && world.z>17.55 && world.z<26.85)continue;
                     let cost=1;
                     if(returnTrip)cost+=returnDoorPenalty(world.x,world.z);
                     if(avoidOccupied){
                         for(const other of otherRobots){
                             const d=f5Dist(world,other.mesh.position);
                             if(d<SAFE_GAP+.12)cost+=28;
                             else if(d<SAFE_GAP+.65)cost+=7;
                             else if(d<SAFE_GAP+1.3)cost+=1.7;
                             // Bonus avoidance for robot's immediate forward occupancy.
                             if(other.speed>.1 && d<2.5){
                                 const vec=f5HeadingVector(other);
                                 const future={x:other.mesh.position.x+vec.x*other.speed*.65,
                                               z:other.mesh.position.z+vec.z*other.speed*.65};
                                 if(f5Dist(world,future)<SAFE_GAP+.25)cost+=8;
                             }
                         }
                     }
                     const value=gScore[k]+cost;
                     if(value<gScore[nk]){
                         gScore[nk]=value;parent[nk]=k;
                         push({n:v,f:value+h(v)});
                     }
                 }
             }
             if(!found)return [];
             const path=[];
             for(let k=targetKey;k!==-1;k=parent[k]){
                 const x=k%GRID_DIM,z=Math.floor(k/GRID_DIM);
                 path.push(f5World({x,z}));
             }
             return path.reverse();
         }
         function f5Plan(startWorld,goalWorld,exceptId,avoidOccupied=false){
             return f5PlanCost(startWorld,goalWorld,exceptId,avoidOccupied,false);
         }
         // The doors are operation-only areas. On RETURN, avoid both restricted
         // regions even if a geometrically shorter path cuts through their frontage.
         // If the return starts inside one zone after a delivery, a high *penalty*
         // (not an absolute ban) allows escape and avoids navigation deadlocks.
         function returnDoorPenalty(x,z){
             function zonePenalty(door){
                 const dx=Math.abs(x-door.x), dz=Math.abs(z-door.z);
                 if(dx<3.05 && dz<3.8) return 85;
                 if(dx<4.1 && dz<4.5) return 6;
                 return 0;
             }
             return zonePenalty(INBOUND_DOOR_POS)+zonePenalty(DESPATCH_DOOR_POS);
         }
         function f5ReturnPlan(startWorld,goalWorld,exceptId,avoidOccupied){
             return f5PlanCost(startWorld,goalWorld,exceptId,avoidOccupied,true);
         }
         function planRobotDockReturn(r,avoidOccupied=true,startOverride=null){
             const approach={x:DOCK_APPROACH_X,z:r.bay.z};
             let path=f5ReturnPlan(startOverride||r.mesh.position,approach,r.id,avoidOccupied);
             if(!path.length)return [];
             // Final straight, perpendicular-to-wall entry; DO NOT turn while moving.
             path.push({x:DOCK_APPROACH_X,z:r.bay.z});
             path.push({x:r.bay.x,z:r.bay.z});
             return path.filter((p,i,a)=>i===0 || f5Dist(p,a[i-1])>.02);
         }
         function planCurrentFleetRoute(r,avoidOccupied=true){
             if(r.phase==='RETURN')return planRobotDockReturn(r,avoidOccupied);
             const goal=r.phase==='PICK'?r.target:r.drop;
             if(!goal)return [];
             const a=r.mesh.position;
             // Leaving from a charging bay is a deliberate +X movement. The
             // regular graph excludes the charging strip for through traffic.
             if(f5Dist(a,r.bay)<.30 && a.x<DOCK_APPROACH_X-.35){
                 const exit={x:DOCK_APPROACH_X,z:r.bay.z};
                 const after=f5Plan(exit,goal,r.id,avoidOccupied);
                 return after.length?[exit,...after]:[];
             }
             return f5Plan(a,goal,r.id,avoidOccupied);
         }
         function beginDockTurn(r){
             r.mesh.position.x=r.bay.x;
             r.mesh.position.z=r.bay.z;
             r.mesh.rotation.y=DOCK_ENTRY_HEADING;
             r.phase='DOCK_TURN';r.state='ĐÃ VÀO Ô - ĐANG QUAY 180°';
             r.speed=0;r.dockTurnProgress=0;r.path=[];r.index=0;
             f5ClearRoute(r);
         }
         function f5MakeRobot(i){
             const g=new THREE.Group();
             const body=new THREE.Mesh(new THREE.BoxGeometry(.62,.20,.78),
                 new THREE.MeshStandardMaterial({color:fleet5Colors[i],metalness:.12,roughness:.72}));
             body.position.y=.16;g.add(body);
             const top=new THREE.Mesh(new THREE.BoxGeometry(.22,.08,.22),
                 new THREE.MeshBasicMaterial({color:0xf8fafc}));
             top.position.y=.30;g.add(top);
             // Directional nose marker: the bright triangle is the robot's FRONT (+Z).
             const nose=new THREE.Mesh(new THREE.ConeGeometry(.16,.35,3),
                 new THREE.MeshBasicMaterial({color:0xfef08a}));
             nose.rotation.x=Math.PI/2;
             nose.position.set(0,.30,.43);
             g.add(nose);
             scene.add(g);return g;
         }
         function f5ClearRoute(r){
             if(r.line){scene.remove(r.line);r.line.geometry.dispose();r.line.material.dispose();r.line=null;}
         }
         function f5RenderRoute(r){
             f5ClearRoute(r);
             if(r.path.length<2)return;
             const pts=[r.mesh.position.clone(),...r.path.slice(r.index).map(p=>new THREE.Vector3(p.x,.06,p.z))];
             const geo=new THREE.BufferGeometry().setFromPoints(pts);
             r.line=new THREE.Line(geo,new THREE.LineBasicMaterial({color:r.color,transparent:true,opacity:.85}));
             scene.add(r.line);
         }

         function f5ClearVisuals(){
             fleet5.forEach(r=>{
                 f5ClearRoute(r);
                 if(r.cargo){r.mesh.remove(r.cargo);r.cargo.geometry.dispose();r.cargo.material.dispose();}
                 scene.remove(r.mesh);
             });
             fleet5=[];fleet5Running=false;fleet5Clock=0;
         }
         // Single fleet model: all five robots share the same mesh, state machine, controls.
         function initFiveFleet(){
             f5ClearVisuals();
             CHARGING_BAYS.forEach((bay,i)=>{
                 const mesh=f5MakeRobot(i);
                 mesh.position.set(bay.x,.02,bay.z);
                 mesh.rotation.y=Math.PI/2;  // Front points +X: away from left wall, toward aisle.
                 fleet5.push({id:i,mesh,path:[],index:0,line:null,speed:0,
                     state:'CHỜ LỆNH',target:null,drop:null,phase:'IDLE',bay,
                     color:fleet5Colors[i],wait:0,travel:0,distance:0,completed:0,
                     cargo:null,battery:100,chargeRemaining:0,taskStart:0,
                     activeTask:null,blockedNow:false,jobQueue:[],
                     dockTurnProgress:0,nextRetryAt:0,scanProgress:0,
                      scanStartHeading:mesh.rotation.y,scanReturnPhase:null,
                      lastMovingHeading:mesh.rotation.y,lastReplanAt:-10,
                      lastProgressAt:0,priorityBoost:0,
                      lastRouteTryAt:-10,nextScanAllowedAt:12,
                      lastDecisionAt:-10,cachedPredictive:null,
                      steeringUntil:0,blockedSince:null});
             });
             f5RefreshTable();
         }
         function fleetRobotByChoice(){
             const el=document.getElementById('fleet-select-robot');
             return fleet5[Number(el?.value||0)]||fleet5[0];
         }
         function fleetBusy(r){
             return ['PICK','DROP','RETURN','CHARGE'].includes(r.phase);
         }
         function pathDistance(origin,path){
             if(!path||!path.length)return Infinity;
             let distance=0,point=origin;
             for(const next of path){distance+=f5Dist(point,next);point=next;}
             return distance;
         }
         function f5Assign(r,slot,dropLoc){
             if(!r||!slot||slot.isEmpty||fleetBusy(r)||reservedSlotIds.has(slot.id))return false;
             const dropSlot=slotsData.find(s=>s.locationCode===dropLoc);
             // Shared dock coordinates made five robots converge on EXACTLY the
             // same point, forming a permanent three-way deadlock. Represent each
             // door by five separate unloading berths along its frontage. These
             // are real geometry locations, not additional inventory records.
             const inbound=dropLoc==='INBOUND';
             const dock=inbound?INBOUND_DOOR_POS:DESPATCH_DOOR_POS;
             const drop=dropSlot || {x:dock.x+(r.id-2.5)*1.40,z:26.65,y:.15};
             const previousPhase=r.phase,previousTarget=r.target;
             r.phase='PICK';r.target=slot;
             let path=planCurrentFleetRoute(r,true);
             if(!path.length)path=planCurrentFleetRoute(r,false);
             r.phase=previousPhase;r.target=previousTarget;
             if(!path.length)return false;
             const fromShelf=f5Plan(slot,drop,r.id,false);
             const predictedDistance=pathDistance(r.mesh.position,path)+pathDistance(slot,fromShelf);
             reservedSlotIds.add(slot.id);
             r.activeTask={sku:String(slot.sku||slot.productId),slotId:slot.id,
                 start:fleet5Clock,startDistance:r.distance,
                 estimatedSeconds:Number.isFinite(predictedDistance)?predictedDistance/MAX_SPEED:null,
                 conflictCount:0,collisionCount:0};
             r.blockedNow=false;r.priorityBoost=0;
             r.blockedSince=null;r.lastRouteTryAt=fleet5Clock;
             r.nextScanAllowedAt=fleet5Clock+SCAN_COOLDOWN;
             r.cachedPredictive=null;
             r.target=slot;r.drop=drop;r.phase='PICK';r.state='ĐI LẤY HÀNG';
             r.index=0;r.path=path;r.wait=0;r.speed=0;r.taskStart=fleet5Clock;
             f5RenderRoute(r);fleet5Running=true;return true;
         }
         function completeFleetTask(r){
             const t=r.activeTask;
             if(!t)return;
             taskHistory.push({robot:`Robot ${r.id+1}`,sku:t.sku,
                 estimated:t.estimatedSeconds,
                 actual:fleet5Clock-t.start,
                 meters:r.distance-t.startDistance,
                 conflicts:t.conflictCount,collisions:t.collisionCount});
             reservedSlotIds.delete(t.slotId);
             r.activeTask=null;
             renderTaskHistory();
             renderPendingDispatches();
         }
         function renderTaskHistory(){
             const body=document.getElementById('task-history-body');
             const count=document.getElementById('history-count');
             if(count)count.textContent=`${taskHistory.length} đơn hoàn thành`;
             if(!body)return;
             const fmt=n=>Number.isFinite(n)?n.toFixed(1):'Không có đường';
             body.innerHTML=taskHistory.length?taskHistory.map((e,i)=>`<tr style="border-bottom:1px solid #e2e8f0">
                 <td class="p-3">${i+1}</td><td class="p-3 font-bold">${e.robot}</td>
                 <td class="p-3 font-mono">${escapeHTML(e.sku)}</td>
                 <td class="p-3 text-right">${fmt(e.estimated)} s</td>
                 <td class="p-3 text-right">${fmt(e.actual)} s</td>
                 <td class="p-3 text-right">${fmt(e.meters)}</td>
                 <td class="p-3 text-right">${e.conflicts}</td>
                 <td class="p-3 text-right">${e.collisions}</td></tr>`).join(''):
                 '<tr><td colspan="8" class="text-center text-slate-500 p-5">Chưa có lượt giao hàng nào hoàn tất.</td></tr>';
         }
         function escapeHTML(value){return String(value).replace(/[&<>"']/g,c=>({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[c]));}
         function downloadTaskHistoryCSV(){
             const head=['Robot','SKU','Thoi_gian_du_tinh_s','Thoi_gian_thuc_te_s','Quang_duong_m','Xung_dot','Va_cham'];
             const quote=v=>'"'+String(v??'').replace(/"/g,'""')+'"';
             const lines=taskHistory.map(t=>[t.robot,t.sku,t.estimated??'',t.actual.toFixed(2),t.meters.toFixed(2),t.conflicts,t.collisions].map(quote).join(','));
             const blob=new Blob(['\uFEFF'+[head.join(','),...lines].join('\\n')],{type:'text/csv;charset=utf-8'});
             const url=URL.createObjectURL(blob),a=document.createElement('a');
             a.href=url;a.download='lich_su_lay_hang_robot.csv';document.body.appendChild(a);a.click();a.remove();
             setTimeout(()=>URL.revokeObjectURL(url),1000);
         }
         function getDropLabel(drop){
             if(drop==='DESPATCH') return 'Cửa xuất';
             if(drop==='INBOUND') return 'Cửa nhập';
             return drop;
         }
         function renderPendingDispatches(){
             const body=document.getElementById('pending-dispatch-body');
             const badge=document.getElementById('pending-count-badge');
             if(!body || !badge) return;
             badge.textContent=`${pendingDispatches.length} lệnh`;
             if(!pendingDispatches.length){
                 body.innerHTML='<tr><td colspan="4" class="py-3 text-center text-slate-400">Chưa có lệnh chờ</td></tr>';
                 return;
             }
             body.innerHTML=pendingDispatches.map((cmd,idx)=>`<tr style="border-bottom:1px solid #e2e8f0"><td class="py-1 pr-1 font-mono text-slate-700">${escapeHTML(cmd.sku)}</td><td class="py-1 pr-1 text-red-600 font-bold">Robot ${cmd.robotId+1}</td><td class="py-1 pr-1">${escapeHTML(getDropLabel(cmd.dropLoc))}</td><td class="py-1 text-right"><button onclick="removePendingDispatch(${idx})" class="text-red-600 font-bold">✕</button></td></tr>`).join('');
         }
         function removePendingDispatch(index){ pendingDispatches.splice(index,1); renderPendingDispatches(); }
         function clearPendingDispatches(){ pendingDispatches=[]; renderPendingDispatches(); const feedback=document.getElementById('dispatch-feedback'); if(feedback)feedback.textContent='Đã xóa toàn bộ dòng chờ phát lệnh.'; }
         function queuedCountForRobot(robotId){
             return fleet5[robotId]?.jobQueue?.length||0;
         }
         function startNextFleetOrder(r){
             if(!r || r.phase==='PICK'||r.phase==='DROP'||r.phase==='DOCK_TURN')return false;
             if(r.phase==='CHARGE')return false;
             while(r.jobQueue.length){
                 const cmd=r.jobQueue[0];
                 const slot=slotsData.find(s=>s.id===cmd.slotId);
                 if(!slot || slot.isEmpty || reservedSlotIds.has(cmd.slotId)){
                     r.jobQueue.shift(); queuedSlotIds.delete(cmd.slotId);
                     continue;
                 }
                 // Keep the queued-slot reservation until the assignment is committed.
                 if(f5Assign(r,slot,cmd.dropLoc)){
                     r.jobQueue.shift();queuedSlotIds.delete(cmd.slotId);
                     return true;
                 }
                 r.phase='WAIT_NEXT';r.state='CHỜ LỐI ĐẾN ĐƠN TIẾP';
                 r.nextRetryAt=fleet5Clock+2;
                 return false;
             }
             return false;
         }
         function addPendingDispatch(){
             const slot=slotsData.find(s=>String(s.id)===document.getElementById('agv-select-item').value);
             const robotId=Number(document.getElementById('fleet-select-robot').value);
             const dropLoc=document.getElementById('agv-select-drop').value;
             const feedback=document.getElementById('dispatch-feedback');
             if(!slot || slot.isEmpty){ if(feedback)feedback.textContent='Vui lòng chọn SKU hợp lệ còn hàng trong kho.'; return; }
             if(reservedSlotIds.has(slot.id) || queuedSlotIds.has(slot.id) || pendingDispatches.some(cmd=>cmd.slotId===slot.id)){ if(feedback)feedback.textContent=`SKU ${slot.sku} đã có robot khác nhận hoặc đang nằm trong danh sách chờ.`; return; }
             pendingDispatches.push({slotId:slot.id,sku:slot.sku,robotId,dropLoc});
             renderPendingDispatches();
             const nth=pendingDispatches.filter(c=>c.robotId===robotId).length + queuedCountForRobot(robotId);
             if(feedback)feedback.textContent=`Đã thêm đơn thứ ${nth} cho Robot ${robotId+1}: SKU ${slot.sku} → ${getDropLabel(dropLoc)}.`;
         }
         function dispatchPendingQueue(){
             const feedback=document.getElementById('dispatch-feedback');
             if(!pendingDispatches.length){ if(feedback)feedback.textContent='Chưa có dòng chờ nào để phát lệnh.'; return; }
             let accepted=0,started=0;
             for(const cmd of pendingDispatches){
                 const slot=slotsData.find(s=>s.id===cmd.slotId), r=fleet5[cmd.robotId];
                 if(!slot || slot.isEmpty || !r || reservedSlotIds.has(cmd.slotId) || queuedSlotIds.has(cmd.slotId))continue;
                 r.jobQueue.push({...cmd});queuedSlotIds.add(cmd.slotId);accepted++;
             }
             pendingDispatches=[];
             fleet5.forEach(r=>{
                 if(!r.jobQueue.length)return;
                 if(r.phase==='IDLE'||r.phase==='DONE'||r.phase==='WAIT_NEXT'){
                     if(startNextFleetOrder(r))started++;
                 }else if(r.phase==='RETURN'){
                     // A newly issued batch cancels an unnecessary return journey.
                     r.phase='WAIT_NEXT';r.path=[];r.index=0;f5ClearRoute(r);
                     if(startNextFleetOrder(r))started++;
                 }
             });
             renderPendingDispatches();f5RefreshTable();
             if(feedback)feedback.textContent=`✅ Đã tiếp nhận ${accepted} đơn: ${started} robot bắt đầu; các đơn còn lại nằm trong hàng đợi riêng của từng robot. Chỉ quay về sạc sau khi xong toàn bộ đơn.`;
         }
         function f5AssignSelected(){
             addPendingDispatch();
             dispatchPendingQueue();
         }
         function startFiveFleet(){
             const selected=slotsData.find(s=>String(s.id)===document.getElementById('agv-select-item').value);
             const candidates=slotsData.filter(s=>!s.isEmpty && s.quantity>0)
                 .sort((a,b)=>a.daysLeft-b.daysLeft);
             if(!candidates.length){alert('Kho không có sản phẩm hợp lệ.');return;}
             const drop=document.getElementById('agv-select-drop').value;
             let used=new Set(),successes=0;
             fleet5.forEach((r,i)=>{
                 if(fleetBusy(r))return;
                 let slot=i===0&&selected&&!used.has(selected.id)?selected:null;
                 if(!slot)slot=candidates.find(c=>!used.has(c.id) && c.id!==(selected?.id)) || candidates.find(c=>!used.has(c.id));
                 if(slot && f5Assign(r,slot,drop)){successes++;used.add(slot.id);}
             });
             // Multi-dispatch is not exposed in the streamlined UI.
             f5RefreshTable();
         }
         function fleetReturn(r){
             if(!r)return false;
             if(r.activeTask){reservedSlotIds.delete(r.activeTask.slotId);r.activeTask=null;}
             if(r.cargo){r.mesh.remove(r.cargo);r.cargo.geometry.dispose();r.cargo.material.dispose();r.cargo=null;}
             r.target=null;r.drop=null;r.speed=0;r.wait=0;
             if(f5Dist(r.mesh.position,r.bay)<.12){beginDockTurn(r);return true;}
             r.phase='RETURN';
             // Returns use a door-aware route, then approach the charging pad at 90°.
             r.path=planRobotDockReturn(r,true);
             if(!r.path.length)r.path=planRobotDockReturn(r,false);
             r.index=0;r.state=r.path.length?'TRỞ VỀ SẠC - TRÁNH CỬA':'CHỜ ĐƯỜNG VỀ SẠC';
             f5RenderRoute(r);fleet5Running=true;
             return Boolean(r.path.length);
         }
         function clearRobotQueuedOrders(r){
             for(const cmd of r.jobQueue)queuedSlotIds.delete(cmd.slotId);
             r.jobQueue=[];
         }
         function returnSelectedRobot(){
             const r=fleetRobotByChoice();
             clearRobotQueuedOrders(r);fleetReturn(r);f5RefreshTable();
         }
         function returnAllRobots(){
             fleet5.forEach(r=>{clearRobotQueuedOrders(r);fleetReturn(r);});
             f5RefreshTable();
         }
         // Predictive multi-agent controller. Relative velocity and Closest Point
         // of Approach (CPA) estimate conflicts up to CONFLICT_HORIZON seconds ahead.
         // This avoids a common flaw of only checking current robot separation.
         function f5HeadingVector(r){
             if(['PICK','DROP','RETURN'].includes(r.phase) && r.path.length){
                 const b=r.path[Math.min(r.index,r.path.length-1)];
                 const dx=b.x-r.mesh.position.x,dz=b.z-r.mesh.position.z;
                 const dist=Math.hypot(dx,dz);
                 if(dist>.08)return {x:dx/dist,z:dz/dist};
             }
             return {x:Math.sin(r.mesh.rotation.y),z:Math.cos(r.mesh.rotation.y)};
         }
         function f5PredictiveControl(r,direction){
             const a=r.mesh.position;
             const len=Math.hypot(direction.x,direction.z);
             if(len<1e-8)return {limit:0,reason:'NONE',risk:0};
             const ux=direction.x/len,uz=direction.z/len;
             let limit=MAX_SPEED,risk=0,reason='CLEAR';
             for(const other of fleet5){
                 if(other.id===r.id)continue;
                 const p=other.mesh.position;
                 const rx=p.x-a.x,rz=p.z-a.z;
                 const dist=Math.hypot(rx,rz);
                 if(dist>SENSOR_RANGE+1.2)continue;
                 const forward=rx*ux+rz*uz;
                 const lateral=Math.abs(rx*uz-rz*ux);
                 // Distinguish diagonal/side robots from true frontal blockage:
                 // a generous rectangular sensor used to STOP both robots at
                 // perpendicular crossings even with >1m actual separation.
                 const isDirectlyAhead=forward>.12 && lateral<SAFE_GAP*.82;
                 if(isDirectlyAhead){
                     // Project the circular clearance envelope into the travel
                     // direction instead of treating diagonals as head-on walls.
                     const longitudinal=Math.sqrt(Math.max(0,(SAFE_GAP+.08)**2-lateral*lateral));
                     const clearance=Math.max(0,forward-longitudinal-.04);
                     const cap=Math.sqrt(2*BRAKE*clearance);
                     limit=Math.min(limit,cap);
                     if(cap<MAX_SPEED)risk++;
                     continue;
                 }
                 if(dist<SIDE_RANGE){
                     // A side/rear neighbor is a slowdown, NOT a reason to scan.
                     const cap=Math.max(.45,Math.min(MAX_SPEED,
                         .45+(dist-SAFE_GAP)*1.20));
                     limit=Math.min(limit,cap);
                     reason='SIDE_NEIGHBOR';
                 }
                 // For converging moving robots use predicted closest approach.
                 // Do not react to a stationary side robot as a head-on hazard.
                 if(other.speed>.12 && forward>0 &&
                    lateral<2.3 && dist<MAX_SPEED*CONFLICT_HORIZON+1){
                     const ov=f5HeadingVector(other);
                     const closingSpeed=Math.min(MAX_SPEED,Math.max(.25,r.speed));
                     const rvx=ov.x*other.speed-ux*closingSpeed;
                     const rvz=ov.z*other.speed-uz*closingSpeed;
                     const rv2=rvx*rvx+rvz*rvz;
                     const t=rv2>1e-8 ? Math.max(0,Math.min(CONFLICT_HORIZON,
                         -(rx*rvx+rz*rvz)/rv2)):0;
                     const closest=Math.hypot(rx+rvx*t,rz+rvz*t);
                     if(closest<SAFE_GAP+.16){
                         limit=Math.min(limit,Math.max(.25,
                           Math.sqrt(2*BRAKE*Math.max(0,dist-SAFE_GAP-.25))));
                         reason='CPA_CONFLICT';risk++;
                     }
                 }
             }
             return {limit:Math.max(0,Math.min(MAX_SPEED,limit)),reason,risk};
         }
         function f5WrapAngle(x){return Math.atan2(Math.sin(x),Math.cos(x));}
         function f5TurnToward(r,heading,dt){
             const diff=f5WrapAngle(heading-r.mesh.rotation.y);
             const step=Math.max(-ANGULAR_RATE*dt,Math.min(ANGULAR_RATE*dt,diff));
             r.mesh.rotation.y=f5WrapAngle(r.mesh.rotation.y+step);
             return Math.abs(diff);
         }
         function f5TryRouteOnce(r,allowFallback=false){
             if(fleet5Clock-r.lastRouteTryAt<REPLAN_INTERVAL)return false;
             r.lastRouteTryAt=fleet5Clock;
             let candidate=planCurrentFleetRoute(r,true);
             if(!candidate.length && allowFallback)candidate=planCurrentFleetRoute(r,false);
             if(!candidate.length)return false;
             // Keep the current route if the replacement makes identical progress;
             // otherwise accept the new route even if its initial cell is shared.
             const newNext=candidate.find(p=>f5Dist(p,r.mesh.position)>.16);
             if(!newNext)return false;
             const oldNext=r.path.slice(r.index).find(p=>f5Dist(p,r.mesh.position)>.16);
             const same=oldNext && f5Dist(oldNext,newNext)<.13;
             if(same && r.path.length)return false;
             r.path=candidate;r.index=0;r.wait=0;r.blockedSince=null;
             r.cachedPredictive=null;
             r.state='ĐÃ TÌM ĐƯỜNG VÒNG';f5RenderRoute(r);
             return true;
         }
         // Local yield planning: choose a nearby safe lay-by and continue to the
         // original goal. Invoked only after a real, sustained no-progress interval.
         function f5TryCooperativeDetour(r){
             const a=r.mesh.position;
             const objective=r.phase==='RETURN'?r.bay:r.phase==='PICK'?r.target:r.drop;
             if(!objective)return false;
             const oldNext=r.path.slice(r.index).find(p=>f5Dist(p,a)>.20);
             const candidates=[];
             for(const radius of [1.5,2.5,3.5])for(let k=0;k<8;k++){
                 const theta=k*Math.PI/4;
                 const proposed={x:a.x+radius*Math.cos(theta),z:a.z+radius*Math.sin(theta)};
                 if(proposed.x<.6||proposed.z<.6||proposed.x>WORLD_SIZE-.6||proposed.z>WORLD_SIZE-.6)continue;
                 const node=f5Nearest(proposed.x,proposed.z);
                 if(!node)continue;
                 const via=f5World(node);
                 if(f5Dist(via,proposed)>.4||f5Dist(via,a)<1.1)continue;
                 if(fleet5.some(o=>o.id!==r.id && f5Dist(o.mesh.position,via)<SAFE_GAP+0.65))continue;
                 if(r.phase==='RETURN' && returnDoorPenalty(via.x,via.z)>10)continue;
                 const toVia=f5PlanCost(a,via,r.id,true,r.phase==='RETURN');
                 if(!toVia.length)continue;
                 const fromVia=r.phase==='RETURN'
                     ?planRobotDockReturn(r,true,via)
                     :f5Plan(via,objective,r.id,true);
                 if(!fromVia.length)continue;
                 const newFirst=toVia.find(p=>f5Dist(p,a)>.15);
                 if(!newFirst||oldNext&&f5Dist(oldNext,newFirst)<.13)continue;
                 const route=[...toVia,...fromVia.slice(1)];
                 const score=pathDistance(a,route)+radius*.2;
                 candidates.push({route,score});
                 if(candidates.length>=5)break;
             }
             if(!candidates.length)return false;
             candidates.sort((x,y)=>x.score-y.score);
             r.path=candidates[0].route;r.index=0;r.cachedPredictive=null;
             r.blockedSince=null;r.state='RẼ LỐI TRÁNH ÙN TẮC';
             r.lastRouteTryAt=fleet5Clock;f5RenderRoute(r);
             return true;
         }
         function f5StartScan(r){
             if(r.phase==='SCAN_360' || fleet5Clock<r.nextScanAllowedAt)return false;
             // Prefer immediate navigation to an avoid-occupied route over spinning.
             if(f5TryRouteOnce(r,false))return false;
             // Avoid endless spinning when another robot is simply passing nearby.
             const nearbyMoving=fleet5.some(other=>other.id!==r.id &&
                 other.speed>.10 && f5Dist(other.mesh.position,r.mesh.position)<2.5);
             if(nearbyMoving){r.state='NHƯỜNG ROBOT ĐANG ĐI QUA';return false;}
             r.scanReturnPhase=r.phase;
             r.scanStartHeading=r.mesh.rotation.y;
             r.scanProgress=0;
             r.phase='SCAN_360';r.speed=0;r.wait=0;
             r.nextScanAllowedAt=fleet5Clock+SCAN_COOLDOWN;
             r.state='BẾ TẮC LÂU - QUÉT 360° MỘT LẦN';
             return true;
         }
         function f5CandidateRoute(r,candidate){
             const goal=r.scanReturnPhase==='RETURN' || r.phase==='RETURN';
             const first=f5Plan(r.mesh.position,candidate,r.id,true);
             if(!first.length)return [];
             let second;
             if(goal)second=planRobotDockReturn(r,true,candidate);
             else {
                 const destination=(r.scanReturnPhase==='PICK'||r.phase==='PICK')?r.target:r.drop;
                 second=destination?f5Plan(candidate,destination,r.id,true):[];
             }
             if(!second.length)return [];
             return [...first,...second.slice(1)].filter((p,i,a)=>i===0||f5Dist(p,a[i-1])>.06);
         }
         function f5ScanAndReplan(r){
             // Evaluate 16 headings around the robot (360 degrees), two radii.
             // Penalize narrow passages and directions of predicted robot movement.
             const a=r.mesh.position, originalPhase=r.scanReturnPhase;
             let best=null,bestScore=Infinity;
             const goal=originalPhase==='RETURN'?r.bay:originalPhase==='PICK'?r.target:r.drop;
             if(!goal)return false;
             const direct=f5Dist(a,goal);
             for(const radius of [1.5,2.5]){
                 for(let k=0;k<16;k++){
                     const angle=(k*2*Math.PI/16);
                     const px=a.x+Math.sin(angle)*radius,pz=a.z+Math.cos(angle)*radius;
                     if(px<.5||px>WORLD_SIZE-.5||pz<.5||pz>WORLD_SIZE-.5)continue;
                     const node=f5Nearest(px,pz);
                     if(!node)continue;
                     const candidate=f5World(node);
                     if(f5Dist(candidate,{x:px,z:pz})>.55)continue;
                     // Any candidate must have clearance from existing and forecast robots.
                     if(fleet5.some(other=>{
                         if(other.id===r.id)return false;
                         const pos=other.mesh.position,heading=f5HeadingVector(other);
                         return [0,.8,1.5].some(t=>f5Dist(candidate,{
                             x:pos.x+heading.x*other.speed*t,
                             z:pos.z+heading.z*other.speed*t})<SAFE_GAP+.30);
                     }))continue;
                     const route=f5CandidateRoute(r,candidate);
                     if(route.length<2)continue;
                     const routeLength=pathDistance(a,route);
                     // Side stepping or reversing may be necessary in a deadlock.
                     const angleChange=Math.abs(f5WrapAngle(angle-r.scanStartHeading));
                     const score=routeLength+Math.max(0,f5Dist(candidate,goal)-direct)*.6+
                         angleChange*.25+radius*.15 + (k===0?.2:0);
                     if(score<bestScore){bestScore=score;best=route;}
                 }
             }
             if(best){
                 r.path=best;r.index=0;r.wait=0;r.lastReplanAt=fleet5Clock;
                 f5RenderRoute(r);return true;
             }
             return false;
         }
         // Predictive speed control. Forward corridor may force a stop; side/rear
         // obstacles only lower cruise speed. This is NOT collision avoidance alone:
         // f5AllowedMove remains the swept safety check for every movement step.
         function f5SafetySpeed(pos,target,exceptId){
             const vx=target.x-pos.x,vz=target.z-pos.z;
             const len=Math.hypot(vx,vz);
             if(len<1e-8)return 0;
             const ux=vx/len,uz=vz/len;
             let allowed=MAX_SPEED;
             // Dynamic obstacles: other AGVs.
             for(const other of f5OtherPositions(exceptId)){
                 const dx=other.x-pos.x,dz=other.z-pos.z;
                 const dist=Math.hypot(dx,dz);
                 if(dist>SENSOR_RANGE)continue;
                 const ahead=dx*ux+dz*uz;
                 const lateral=Math.abs(dx*uz-dz*ux);
                 if(ahead>0 && lateral<.80){
                     const free=Math.max(0,ahead-SAFE_GAP-0.08);
                     // v^2 <= 2*a*d, so braking distance is respected.
                     const vSafe=Math.sqrt(2*BRAKE*free);
                     allowed=Math.min(allowed,vSafe);
                 } else if(dist<SIDE_RANGE){
                     // Side/rear: cautious, but no stop from sensing alone.
                     const ratio=Math.max(0,Math.min(1,(dist-SAFE_GAP)/(SIDE_RANGE-SAFE_GAP)));
                     allowed=Math.min(allowed,.55+1.2*ratio);
                 }
             }
             // Static obstacles (rack bodies): sample only within the forward corridor.
             // This is conservative near shelf corners without stopping due to side racks.
             for(const bbox of obstacleBoxes){
                 const x=Math.max(bbox.min.x,Math.min(pos.x,bbox.max.x));
                 const z=Math.max(bbox.min.z,Math.min(pos.z,bbox.max.z));
                 const close=Math.hypot(x-pos.x,z-pos.z);
                 if(close>SENSOR_RANGE)continue;
                 // Sample the heading, not the entire obstacle box, to avoid braking
                 // for long racks running parallel to the robot.
                 let forwardGap=Infinity;
                 for(let d=.45;d<=Math.min(SENSOR_RANGE,4.4);d+=.2){
                     const px=pos.x+ux*d,pz=pos.z+uz*d;
                     if(px>bbox.min.x-.24 && px<bbox.max.x+.24 &&
                        pz>bbox.min.z-.24 && pz<bbox.max.z+.24){forwardGap=d;break;}
                 }
                 if(Number.isFinite(forwardGap)){
                     // A future path cell may terminate next to a shelf; slow down
                     // but leave final wall blocking to navigation rather than sensor.
                     const vSafe=Math.sqrt(2*BRAKE*Math.max(0,forwardGap-.25));
                     allowed=Math.min(allowed,Math.max(.25,vSafe));
                 } else if(close<.9){
                     allowed=Math.min(allowed,.9);
                 }
             }
             return Math.max(0,Math.min(MAX_SPEED,allowed));
         }
         // Continuous swept collision: no frame can move one robot through another.
         function f5AllowedMove(from,to,exceptId,gap=SAFE_GAP){
             for(const other of f5OtherPositions(exceptId)){
                 const vx=to.x-from.x,vz=to.z-from.z;
                 const len2=vx*vx+vz*vz;
                 const u=len2>0?Math.max(0,Math.min(1,((other.x-from.x)*vx+(other.z-from.z)*vz)/len2)):0;
                 const closest={x:from.x+vx*u,z:from.z+vz*u};
                 if(f5Dist(closest,other)<gap)return false;
             }
             return true;
         }

         function f5Advance(r,delta){
             if(r.phase==='DOCK_TURN'){
                 r.speed=0;
                 r.dockTurnProgress=Math.min(Math.PI,r.dockTurnProgress+DOCK_TURN_RATE*delta);
                 r.mesh.rotation.y=DOCK_ENTRY_HEADING+r.dockTurnProgress;
                 r.state='QUAY ĐẦU 180° TẠI TRẠM';
                 if(r.dockTurnProgress>=Math.PI-1e-5){
                     r.mesh.rotation.y=DOCK_EXIT_HEADING;
                     r.phase='CHARGE';r.state='ĐANG SẠC';r.chargeRemaining=10;
                 }
                 return;
             }
             if(r.phase==='SCAN_360'){
                 r.speed=0;
                 const change=Math.min(2*Math.PI-r.scanProgress,SCAN_ANGULAR_RATE*delta);
                 r.scanProgress+=change;
                 r.mesh.rotation.y=f5WrapAngle(r.scanStartHeading+r.scanProgress);
                 r.state=`QUÉT HƯỚNG 360° ${Math.round(r.scanProgress*180/Math.PI)}°`;
                 if(r.scanProgress>=2*Math.PI-1e-4){
                     const previous=r.scanReturnPhase;
                     const planned=f5ScanAndReplan(r);
                     r.phase=previous;r.scanReturnPhase=null;
                     r.wait=0;r.priorityBoost=0;
                     r.lastRouteTryAt=fleet5Clock;
                     r.nextScanAllowedAt=fleet5Clock+SCAN_COOLDOWN;
                     r.blockedSince=planned?null:fleet5Clock;
                     r.state=planned?'ĐÃ CHỌN LỐI - TIẾP TỤC':'CHỜ AN TOÀN - SẼ THỬ LẠI';
                     if(!planned){r.path=[];r.index=0;}
                 }
                 return;
             }
             if(r.phase==='WAIT_NEXT'){
                 r.speed=0;
                 if(fleet5Clock>=r.nextRetryAt){
                     if(!startNextFleetOrder(r) && !r.jobQueue.length)fleetReturn(r);
                 }
                 return;
             }
             if(r.phase==='CHARGE'){
                 r.speed=0;r.mesh.rotation.y=Math.PI/2;
                 r.chargeRemaining=Math.max(0,r.chargeRemaining-delta);
                 r.state=`SẠC PIN (${Math.ceil(r.chargeRemaining)}s)`;
                 if(!r.chargeRemaining){r.battery=100;r.phase='IDLE';r.state='SẴN SÀNG';}
                 return;
             }
             if(!['PICK','DROP','RETURN'].includes(r.phase)){
                 r.speed=0;
                 if(r.phase==='IDLE'&&f5Dist(r.mesh.position,r.bay)<.18)r.mesh.rotation.y=Math.PI/2;
                 return;
             }
             if(!r.path.length){
                 r.speed=0;
                 if(r.phase==='RETURN' && f5Dist(r.mesh.position,r.bay)<.20){
                     beginDockTurn(r);return;
                 }
                 if(r.blockedSince===null)r.blockedSince=fleet5Clock;
                 const waited=fleet5Clock-r.blockedSince;
                 if(f5TryRouteOnce(r,false)){
                     r.state='ĐÃ CÓ LỐI - KHỞI HÀNH';
                     return;
                 }
                 // Only consider a single physical 360 scan after a long true
                 // no-route condition; never scan repeatedly at the dock.
                 if(waited>=4 && fleet5Clock-r.lastReplanAt>=6){
                     r.lastReplanAt=fleet5Clock;
                     f5TryCooperativeDetour(r);
                 }
                 if(waited>=BLOCKED_SCAN_AFTER && fleet5Clock>=r.nextScanAllowedAt){
                     // Physical scan is diagnostic only; navigation continues to
                     // prioritize available detours and precomputed paths.
                     if(!r.path.length)f5StartScan(r);
                 }else{
                     r.state=r.phase==='RETURN'?'CHỜ LỐI VỀ - TRÁNH CỬA':'CHỜ LỐI TRỐNG';
                 }
                 return;
             }
             const a=r.mesh.position;
             while(r.index<r.path.length&&f5Dist(a,r.path[r.index])<.055)r.index++;
             if(r.index>=r.path.length){
                 if(r.phase==='PICK'){
                     r.phase='DROP';r.state='ĐANG GIAO HÀNG';
                     if(r.target&&!r.cargo){
                         r.cargo=new THREE.Mesh(new THREE.BoxGeometry(.34,.24,.34),
                             new THREE.MeshStandardMaterial({color:r.target.colorHex}));
                         r.cargo.position.y=.43;r.mesh.add(r.cargo);
                     }
                     r.path=f5Plan(a,r.drop,r.id,true);
                     if(!r.path.length)r.path=f5Plan(a,r.drop,r.id);
                     r.index=0;r.wait=0;f5RenderRoute(r);
                 } else if(r.phase==='DROP'){
                     r.phase='DONE';r.state='HOÀN THÀNH';r.speed=0;r.completed++;
                     r.battery=Math.max(0,r.battery-10);f5ClearRoute(r);
                     if(r.cargo){r.mesh.remove(r.cargo);r.cargo.geometry.dispose();r.cargo.material.dispose();r.cargo=null;}
                     completeFleetTask(r);
                     if(r.jobQueue.length){
                         // Continue the next order at the current delivery position.
                         if(!startNextFleetOrder(r)){
                             r.phase='WAIT_NEXT';r.state='CHỜ ĐƠN TIẾP';
                             r.nextRetryAt=fleet5Clock+2;
                         }
                     }else{
                         fleetReturn(r); // Return only after the last delivery.
                     }
                 } else if(r.phase==='RETURN'){
                     // Do not rotate to outward heading while still driving into bay.
                     if(f5Dist(a,r.bay)<.16)beginDockTurn(r);
                     else{r.path=[];r.index=0;r.state='ĐANG CĂN CHỈNH VÀO Ô SẠC';}
                 }
                 return;
             }
             const b=r.path[r.index],dist=f5Dist(a,b);
             let targetSpeed=f5SafetySpeed(a,b,r.id);
              if(!r.cachedPredictive || fleet5Clock-r.lastDecisionAt>=SENSOR_DECISION_INTERVAL){
                  r.cachedPredictive=f5PredictiveControl(r,{x:b.x-a.x,z:b.z-a.z});
                  r.lastDecisionAt=fleet5Clock;
              }
              const predictive=r.cachedPredictive;
              targetSpeed=Math.min(targetSpeed,predictive.limit);
              // Collision checker keeps absolute priority. Prediction adjusts
              // speed smoothly but does not force a stopped robot to spin.
             // Path curvature. Brake BEFORE the turn, and accelerate after clearing it.
             const nextPoint=r.path[r.index+1],prevPoint=r.path[r.index-1]||a;
             const curve=(p,q,u)=>Math.abs((q.x-p.x)*(u.z-q.z)-(q.z-p.z)*(u.x-q.x));
             if(nextPoint&&curve(prevPoint,b,nextPoint)>.01){
                 const safeTurnSpeed=Math.sqrt(TURN_SPEED*TURN_SPEED + 2*BRAKE*Math.max(0,dist-.35));
                 targetSpeed=Math.min(targetSpeed,safeTurnSpeed);
             }
             if(r.index>=2){
                 const p=r.path[r.index-2],q=r.path[r.index-1];
                 if(curve(p,q,b)>.01&&f5Dist(a,q)<.75)
                     targetSpeed=Math.min(targetSpeed,TURN_SPEED);
             }
              // 360-degree-capable heading controller: rotate continuously toward
              // each path segment rather than teleporting direction instantly.
              const desiredHeading=Math.atan2(b.x-a.x,b.z-a.z);
              const headingError=f5TurnToward(r,desiredHeading,delta);
              const steeringInPlace=headingError>1.2;
              if(steeringInPlace)targetSpeed=0;
              else if(headingError>.4)targetSpeed=Math.min(targetSpeed,TURN_SPEED*.6);
             // Integrate speed with bounded acceleration/deceleration.
              const dv=targetSpeed-r.speed;
             let speed=Math.max(0,Math.min(MAX_SPEED,r.speed+Math.max(-BRAKE*delta,Math.min(ACCEL*delta,dv))));
             // Do not overshoot a stopped front-facing safety envelope.
             speed=Math.min(speed,targetSpeed);
             let step=Math.min(dist,speed*delta);
             const next={x:dist?a.x+(b.x-a.x)/dist*step:a.x,
                         z:dist?a.z+(b.z-a.z)/dist*step:a.z};
             const preventedByRobot=step>0&&!f5AllowedMove(a,next,r.id);
             if(preventedByRobot){speed=0;step=0;}
             const blockedByRobot=preventedByRobot || (speed===0 &&
                 f5OtherPositions(r.id).some(o=>{
                     const dx=o.x-a.x,dz=o.z-a.z;
                     const gx=b.x-a.x,gz=b.z-a.z;
                     const norm=Math.hypot(gx,gz)||1;
                     return (dx*gx+dz*gz)/norm>0 && Math.hypot(dx,dz)<SAFE_GAP+0.4;
                 }));
             if(r.activeTask && blockedByRobot&&!r.blockedNow)r.activeTask.conflictCount++;
             r.blockedNow=blockedByRobot;
             if(step>0){
                 a.x=next.x;a.z=next.z;
                 r.distance+=step;r.travel+=delta;
                 r.wait=0;r.blockedSince=null;r.priorityBoost=0;
                 r.lastProgressAt=fleet5Clock;
                 // Keep the current predictive decision until its 150 ms expires.
             }else if(steeringInPlace){
                 // Turning to the next waypoint is deliberate motion, not a jam.
                 r.wait=0;r.blockedSince=null;r.priorityBoost=0;
                 r.cachedPredictive=null;
             }else{
                 // A true jam requires both an assigned path and inability to
                 // move forward. Try detouring first; never continuously scan.
                 if(r.blockedSince===null)r.blockedSince=fleet5Clock;
                 const blockedFor=fleet5Clock-r.blockedSince;
                 if(blockedFor>=REPLAN_INTERVAL){
                     if(f5TryRouteOnce(r,false)){
                         r.state='TÌM ĐƯỢC LỐI THOÁT - TIẾP TỤC';
                     }else if(blockedFor>=4.5 && fleet5Clock-r.lastReplanAt>=6){
                         r.lastReplanAt=fleet5Clock;
                         f5TryCooperativeDetour(r);
                     }
                 }
             }
             r.speed=delta>0?step/delta:0;
             if(r.phase==='SCAN_360')return;
             r.state=step>0?(predictive.risk?'CHẠY CHẬM - GIỮ KHOẢNG CÁCH':speed<MAX_SPEED-.15?'ĐIỀU TỐC':r.phase==='RETURN'?'ĐANG VỀ TRẠM':'ĐƯỜNG THOÁNG - ĐI'):(steeringInPlace?'XOAY HƯỚNG ĐỂ TIẾP TỤC':'NHƯỜNG ĐƯỜNG - CHỜ LỐI');
         }

         function f5RefreshTable(){
             const tbody=document.getElementById('fleet-speed-body');
             if(tbody)tbody.innerHTML=fleet5.map(r=>`<tr style="border-bottom:1px solid #e2e8f0">
                 <td style="padding:3px;color:#${r.color.toString(16).padStart(6,'0')};font-weight:bold">Robot ${r.id+1}</td>
                 <td style="padding:3px;text-align:center">${r.speed.toFixed(2)} m/s</td>
                 <td style="padding:3px">${r.state}${r.jobQueue.length?` · ${r.jobQueue.length} đơn chờ`:''}</td>
                 <td style="padding:3px;text-align:right">${r.battery}%</td></tr>`).join('');
             const side=document.getElementById('fleet5-list');
             if(side)side.innerHTML=fleet5.map(r=>`<div style="color:#${r.color.toString(16).padStart(6,'0')}">
                 ● Robot ${r.id+1}: ${r.speed.toFixed(2)} m/s · ${r.state} · Chờ ${r.jobQueue.length} · Pin ${r.battery}%</div>`).join('');
             const selected=fleetRobotByChoice();
             if(selected){
                 const st=document.getElementById('agv-status-detail');
                 if(st)st.textContent=`Robot ${selected.id+1}: ${selected.state}`;
                 const badge=document.getElementById('agv-tab-badge');
                 if(badge)badge.textContent=selected.phase;
                 const batt=document.getElementById('agv-tab-battery-text');
                 if(batt)batt.textContent=`${selected.battery}%`;
                 const steps=document.getElementById('agv-tab-current-steps');
                 if(steps)steps.textContent=`${Math.round(selected.distance/GRID_RES)} bước`;
                 const tm=document.getElementById('agv-tab-current-time');
                 if(tm)tm.textContent=`${selected.travel.toFixed(1)}s`;
                 const total=document.getElementById('agv-tab-total-stats');
                 if(total)total.textContent=`${selected.completed} đơn · ${selected.distance.toFixed(1)} m`;
                 const hud=document.getElementById('agv-hud-status');
                 if(hud)hud.textContent=`🤖 Robot ${selected.id+1}: ${selected.state}`;
                 const battery=document.getElementById('agv-hud-battery');
                 if(battery)battery.textContent=`${selected.battery}%`;
                 const hdSteps=document.getElementById('agv-hud-steps');
                 if(hdSteps)hdSteps.textContent=String(Math.round(selected.distance/GRID_RES));
                 const hdTime=document.getElementById('agv-hud-time');
                 if(hdTime)hdTime.textContent=`${selected.travel.toFixed(1)}s`;
             }
         }
         function f5LogSpeeds(){
             fleetSpeedLog.push({
                 time_s:Number(fleet5Clock.toFixed(1)),
                 ...Object.fromEntries(fleet5.map(r=>['Robot_'+(r.id+1)+'_m_s',Number(r.speed.toFixed(2))]))
             });
             if(fleetSpeedLog.length>7200)fleetSpeedLog.shift();
         }
         function downloadFleetSpeedCSV(){
             if(!fleetSpeedLog.length)f5LogSpeeds();
             const keys=Object.keys(fleetSpeedLog[0]);
             const csv='\uFEFF'+keys.join(',')+'\\n'+fleetSpeedLog.map(row=>keys.map(k=>row[k]).join(',')).join('\\n');
             const a=document.createElement('a');
             const url=URL.createObjectURL(new Blob([csv],{type:'text/csv;charset=utf-8'}));
             a.href=url;a.download=`lich_su_toc_do_robot_1_den_${ROBOT_COUNT}.csv`;document.body.appendChild(a);a.click();a.remove();
             setTimeout(()=>URL.revokeObjectURL(url),1000);
         }
         function updateFiveFleet(delta){
             fleet5Clock+=delta;fleetLogClock+=delta;fleetUIClock+=delta;
             // Deterministic priority, rotated by elapsed time to reduce unfair yielding.
             const priority=Math.floor(fleet5Clock/4)%Math.max(1,fleet5.length);
             for(let k=0;k<fleet5.length;k++)f5Advance(fleet5[(k+priority)%fleet5.length],delta);
             fleet5Running=fleet5.some(r=>['PICK','DROP','RETURN','DOCK_TURN','CHARGE','WAIT_NEXT','SCAN_360'].includes(r.phase));
             if(fleetLogClock>=1){fleetLogClock-=1;f5LogSpeeds();}
             if(fleetUIClock>=.2){fleetUIClock=0;f5RefreshTable();}
         }
         function resetFiveFleet(){
             reservedSlotIds.clear();queuedSlotIds.clear();taskHistory.length=0;pendingDispatches=[];
             initFiveFleet();fleetSpeedLog=[];fleet5Clock=0;fleetLogClock=0;
             renderTaskHistory();renderPendingDispatches();
         }
         // END FLEET MANAGEMENT

        // V12: show runtime failures instead of leaving the simulation black.
        function showRenderError(error,stage){
            const message=(error&&error.message)||String(error);
            const detail=(error&&error.stack)||'';
            console.error('[FEFO 3D] '+stage, error);
            const panel=document.getElementById('render-error-panel');
            if(panel){
                panel.style.display='block';
                panel.textContent='⚠️ KHÔNG THỂ HIỂN THỊ KHO 3D\\nBước: '+stage+'\\nLỗi: '+message+
                    '\\n\\n'+detail+'\\n\\nHãy gửi nội dung lỗi này để sửa đúng nguyên nhân.';
            }
        }
        window.addEventListener('error',e=>{
            if(!document.querySelector('#canvas-container canvas'))
                showRenderError(e.error||e.message,'Tải thư viện hoặc khởi tạo 3D');
        });
        window.addEventListener('unhandledrejection',e=>showRenderError(e.reason,'Lỗi bất đồng bộ'));
        function loadThreeScript(url,expect){
            return new Promise((resolve,reject)=>{
                const tag=document.createElement('script');
                tag.src=url;tag.async=true;
                const timeout=setTimeout(()=>{tag.remove();reject(new Error('Timeout '+url));},9000);
                tag.onload=()=>{clearTimeout(timeout);expect()?resolve():reject(new Error('Script missing API '+url));};
                tag.onerror=()=>{clearTimeout(timeout);reject(new Error('Không tải được '+url));};
                document.head.appendChild(tag);
            });
        }
        async function ensureThreeReady(){
            if(typeof THREE==='undefined'){
                const candidates=[
                    'https://cdn.jsdelivr.net/npm/three@0.128.0/build/three.min.js',
                    'https://unpkg.com/three@0.128.0/build/three.min.js',
                    'https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js'
                ];
                let last;
                for(const url of candidates){
                    try{await loadThreeScript(url,()=>typeof THREE!=='undefined');break;}
                    catch(e){last=e;}
                }
                if(typeof THREE==='undefined')throw new Error('Không tải được Three.js. Kiểm tra kết nối/CDN. '+(last?.message||''));
            }
            // OrbitControls is optional; do not hold the 3D scene waiting for its CDN.
            if(typeof THREE.OrbitControls!=='function')
                console.warn('OrbitControls not loaded. Starting in FPS mode without delay.');
        }
        let warehouseBootStarted=false;
        async function bootWarehouse(){
            if(warehouseBootStarted)return;
            warehouseBootStarted=true;
            const panel=document.getElementById('render-error-panel');
            if(panel){
                panel.style.display='block';
                panel.style.color='#93c5fd';
                panel.textContent='Đang khởi tạo cảnh 3D và kiểm tra thư viện Three.js...';
            }
            try {
                await ensureThreeReady();
                init();
                if(panel)panel.style.display='none';
            }catch(e){
                if(panel)panel.style.color='#fff';
                showRenderError(e,'Khởi tạo kho FEFO');
            }
        }
        function init() {
            const container = document.getElementById('canvas-container');

            scene = new THREE.Scene();
            scene.background = new THREE.Color(0x0f172a);
            scene.fog = new THREE.FogExp2(0x0f172a, 0.015);

            camera = new THREE.PerspectiveCamera(65, window.innerWidth / window.innerHeight, 0.1, 100);
            camera.position.set(14.0, 1.6, 2.5);

            renderer = new THREE.WebGLRenderer({ antialias: true, powerPreference: "high-performance" });
            renderer.setSize(window.innerWidth, window.innerHeight);
            renderer.setPixelRatio(Math.min(window.devicePixelRatio, 2));
            renderer.shadowMap.enabled = true;
            renderer.shadowMap.type = THREE.PCFSoftShadowMap;
            container.appendChild(renderer.domElement);

            // Build scene first. Do not wait for optional WMS hooks to begin drawing.
            setupLights();
            initAStarGrid();
            buildPhotorealisticEnvironmentAndDoors();
            buildContinuousRacksWithFEFOAndEmptySlots();
            createChargingStation3D();
            // The shared five-robot fleet is the only active AGV controller.
            initFiveFleet();
            createFPSPlayerHands();

            if(typeof THREE.OrbitControls==='function'){
                controls=new THREE.OrbitControls(camera,renderer.domElement);
                controls.enableDamping=true;
                controls.dampingFactor=.05;
                controls.target.set(14,2,14);
                controls.enabled=false;
            } else {
                // FPS remains fully operational even when OrbitControls CDN is unavailable.
                controls={enabled:false,target:new THREE.Vector3(14,2,14),update:()=>{}};
                console.warn('OrbitControls unavailable; warehouse renders using FPS mode.');
            }

            // Render one frame before initializing the tablet, so a WMS error
            // can never leave an otherwise valid Three.js scene completely black.
            camera.quaternion.setFromEuler(new THREE.Euler(pitch,yaw,0,'YXZ'));
            renderer.render(scene,camera);
            prevTime=performance.now();
            animate();
            for(const [name,callback] of [
                ['Tablet WMS',()=>{populateTabletDataAndOptions();renderPendingDispatches();}],
                ['Mini-map',setupMiniMapCanvas],
                ['Điều khiển',setupEventListeners],
                ['Đồng hồ',startTabletClock]
            ]){
                try{callback();}catch(error){showRenderError(error,name);}
            }
        }

        function setupLights() {
            const ambient = new THREE.AmbientLight(0xffffff, 0.75);
            scene.add(ambient);

            const sun = new THREE.DirectionalLight(0xfff5f5, 0.85);
            sun.position.set(14, 25, 14);
            sun.castShadow = true;
            scene.add(sun);

            for (let x = 4; x < WORLD_SIZE; x += 6) {
                for (let z = 4; z < WORLD_SIZE; z += 6) {
                    const pLight = new THREE.PointLight(0xffe4e6, 0.35, 12);
                    pLight.position.set(x, 7.5, z);
                    scene.add(pLight);
                }
            }
        }

        function initAStarGrid() {
            gridMap = [];
            for (let x = 0; x < GRID_DIM; x++) {
                gridMap[x] = [];
                for (let z = 0; z < GRID_DIM; z++) {
                    gridMap[x][z] = { x: x, z: z, walkable: true };
                }
            }
        }

        // ==============================================================================
        // TẠO CANVAS TEXTURE SÀN GẠCH EPOXY BUMPMAP & TƯỜNG PANELS SIÊU CHÂN THỰC
        // ==============================================================================
        function createRealisticFloorTextures() {
            // 1. Texture Sàn gạch Epoxy chính
            const canvas = document.createElement('canvas');
            canvas.width = 1024; canvas.height = 1024;
            const ctx = canvas.getContext('2d');

            // 2. BumpMap độ gồ ghề rãnh ron gạch 3D
            const bumpCanvas = document.createElement('canvas');
            bumpCanvas.width = 1024; bumpCanvas.height = 1024;
            const bCtx = bumpCanvas.getContext('2d');

            ctx.fillStyle = '#f1f5f9'; ctx.fillRect(0, 0, 1024, 1024);
            bCtx.fillStyle = '#808080'; bCtx.fillRect(0, 0, 1024, 1024);

            const tileSize = 128;
            for (let x = 0; x < 1024; x += tileSize) {
                for (let y = 0; y < 1024; y += tileSize) {
                    // Vi sai màu gạch nhẹ
                    const shade = Math.floor(Math.random() * 10) - 5;
                    const hexVal = (242 + shade).toString(16);
                    ctx.fillStyle = `#${hexVal}${hexVal}${hexVal}`;
                    ctx.fillRect(x + 2, y + 2, tileSize - 4, tileSize - 4);

                    // Vân bê-tông vi mô (Micro Concrete Grain)
                    for (let i = 0; i < 60; i++) {
                        const nx = x + Math.random() * tileSize;
                        const ny = y + Math.random() * tileSize;
                        ctx.fillStyle = Math.random() > 0.5 ? 'rgba(0,0,0,0.02)' : 'rgba(255,255,255,0.04)';
                        ctx.fillRect(nx, ny, 2, 2);
                    }

                    // Đánh rãnh BumpMap gờ chìm ron gạch
                    bCtx.fillStyle = '#303030';
                    bCtx.fillRect(x, y, tileSize, 3);
                    bCtx.fillRect(x, y, 3, tileSize);
                    bCtx.fillStyle = '#a0a0a0';
                    bCtx.fillRect(x + 4, y + 4, tileSize - 8, tileSize - 8);
                }
            }

            // Vạch sơn chỉ dẫn đường AGV màu vàng
            ctx.fillStyle = 'rgba(234, 179, 8, 0.85)';
            ctx.fillRect(115, 0, 26, 1024);
            ctx.fillRect(510, 0, 26, 1024);
            ctx.fillRect(910, 0, 26, 1024);

            // Vết trầy ma sát bánh xe nhạt (Tire Scuff Marks)
            ctx.strokeStyle = 'rgba(30, 41, 59, 0.08)';
            ctx.lineWidth = 14;
            ctx.beginPath(); ctx.moveTo(128, 0); ctx.lineTo(128, 1024); ctx.stroke();
            ctx.beginPath(); ctx.moveTo(523, 0); ctx.lineTo(523, 1024); ctx.stroke();

            const texture = new THREE.CanvasTexture(canvas);
            texture.wrapS = THREE.RepeatWrapping; texture.wrapT = THREE.RepeatWrapping;
            texture.repeat.set(WORLD_SIZE / 3.5, WORLD_SIZE / 3.5);

            const bumpTexture = new THREE.CanvasTexture(bumpCanvas);
            bumpTexture.wrapS = THREE.RepeatWrapping; bumpTexture.wrapT = THREE.RepeatWrapping;
            bumpTexture.repeat.set(WORLD_SIZE / 3.5, WORLD_SIZE / 3.5);

            return { texture, bumpTexture };
        }

        function createRealisticWallTexture() {
            const canvas = document.createElement('canvas');
            canvas.width = 1024; canvas.height = 512;
            const ctx = canvas.getContext('2d');

            ctx.fillStyle = '#f8fafc'; ctx.fillRect(0, 0, 1024, 512);

            // Mí gờ tấm tôn cách nhiệt Sandwich Panel
            ctx.strokeStyle = '#cbd5e1'; ctx.lineWidth = 3;
            for (let y = 0; y < 512; y += 64) {
                ctx.beginPath(); ctx.moveTo(0, y); ctx.lineTo(1024, y); ctx.stroke();
                ctx.strokeStyle = 'rgba(0,0,0,0.08)'; ctx.lineWidth = 2;
                ctx.beginPath(); ctx.moveTo(0, y+2); ctx.lineTo(1024, y+2); ctx.stroke();
            }

            // Chân tường sơn Đỏ Công Nghiệp
            ctx.fillStyle = '#dc2626'; ctx.fillRect(0, 430, 1024, 82);

            // Biển báo An toàn Kho Hàng dán tường
            ctx.fillStyle = '#0f172a'; ctx.fillRect(180, 160, 240, 130);
            ctx.strokeStyle = '#eab308'; ctx.lineWidth = 6; ctx.strokeRect(185, 165, 230, 120);
            ctx.fillStyle = '#ffffff'; ctx.font = 'bold 18px sans-serif';
            ctx.fillText('AN TOÀN LÀ TRÊN HẾT', 198, 205);
            ctx.font = 'bold 12px sans-serif'; ctx.fillStyle = '#eab308';
            ctx.fillText('CHÚ Ý: ĐỘI AGV 5 ROBOT ĐANG CHẠY', 198, 235);
            ctx.fillText('KHU VỰC FEFO HIGH-BAY', 198, 258);

            const texture = new THREE.CanvasTexture(canvas);
            texture.wrapS = THREE.RepeatWrapping; texture.wrapT = THREE.ClampToEdgeWrapping;
            texture.repeat.set(WORLD_SIZE / 8, 1);

            return texture;
        }

        function buildPhotorealisticEnvironmentAndDoors() {
            const { texture: floorTex, bumpTexture: floorBump } = createRealisticFloorTextures();

            // Sàn nhà kho gạch men Epoxy phản quang siêu thực
            const floorGeo = new THREE.PlaneGeometry(WORLD_SIZE, WORLD_SIZE);
            const floorMat = new THREE.MeshStandardMaterial({
                map: floorTex,
                bumpMap: floorBump,
                bumpScale: 0.025,
                roughness: 0.18,
                metalness: 0.15
            });
            const floor = new THREE.Mesh(floorGeo, floorMat);
            floor.rotation.x = -Math.PI / 2;
            floor.position.set(WORLD_SIZE / 2, 0, WORLD_SIZE / 2);
            floor.receiveShadow = true;
            scene.add(floor);

            // Tường Sandwich Panel công nghiệp
            const wallTex = createRealisticWallTexture();
            const wallMat = new THREE.MeshStandardMaterial({ map: wallTex, roughness: 0.4 });

            const backWall = new THREE.Mesh(new THREE.BoxGeometry(WORLD_SIZE, 8.0, 0.2), wallMat);
            backWall.position.set(WORLD_SIZE/2, 4.0, 0); scene.add(backWall);

            // Gờ chân tường 3D inox/thép chống va đập
            const kickMat = new THREE.MeshStandardMaterial({ color: 0x94a3b8, metalness: 0.9, roughness: 0.2 });
            const kickPlate = new THREE.Mesh(new THREE.BoxGeometry(WORLD_SIZE, 0.15, 0.25), kickMat);
            kickPlate.position.set(WORLD_SIZE/2, 0.08, 0.1); scene.add(kickPlate);

            // Đường ống kỹ thuật 3D chạy dọc tường & trần
            const pipeMat = new THREE.MeshStandardMaterial({ color: 0x64748b, metalness: 0.8, roughness: 0.3 });
            for(let h = 5.0; h <= 6.8; h += 0.8) {
                const pipe = new THREE.Mesh(new THREE.CylinderGeometry(0.06, 0.06, WORLD_SIZE, 16), pipeMat);
                pipe.rotation.z = Math.PI / 2;
                pipe.position.set(WORLD_SIZE/2, h, 0.15);
                scene.add(pipe);
            }

            // Vạch sơn cảnh báo sọc Vàng-Đen dưới sàn quanh Cửa & Trạm Sạc
            createHazardStripesFloor(INBOUND_DOOR_POS.x, INBOUND_DOOR_POS.z - 0.8, 4.2, 1.8);
            createHazardStripesFloor(DESPATCH_DOOR_POS.x, DESPATCH_DOOR_POS.z - 0.8, 4.2, 1.8);
            createHazardStripesFloor(CHARGING_STATION_POS.x, CHARGING_STATION_POS.z, 1.95, 8.4);

            // Cửa cuốn tự động & Cửa nhôm kính
            createRollUpShutterDoor(INBOUND_DOOR_POS.x, INBOUND_DOOR_POS.z, "🚪 CỬA NHẬP (INBOUND)");
            createRollUpShutterDoor(DESPATCH_DOOR_POS.x, DESPATCH_DOOR_POS.z, "🚪 CỬA XUẤT (DESPATCH)");
            createAluminumGlassExitDoor(0.2, 2.0);

            createRoofTrussesAndDecorations();
        }

        function createHazardStripesFloor(x, z, w, d) {
            const canvas = document.createElement('canvas');
            canvas.width = 256; canvas.height = 256;
            const ctx = canvas.getContext('2d');

            ctx.fillStyle = '#eab308'; ctx.fillRect(0, 0, 256, 256);
            ctx.fillStyle = '#0f172a';
            const stripeWidth = 32;
            for(let i = -256; i < 512; i += stripeWidth * 2) {
                ctx.beginPath();
                ctx.moveTo(i, 0); ctx.lineTo(i + stripeWidth, 0);
                ctx.lineTo(i + stripeWidth - 256, 256); ctx.lineTo(i - 256, 256);
                ctx.fill();
            }

            const tex = new THREE.CanvasTexture(canvas);
            const planeGeo = new THREE.PlaneGeometry(w, d);
            const planeMat = new THREE.MeshStandardMaterial({ map: tex, roughness: 0.4 });
            const plane = new THREE.Mesh(planeGeo, planeMat);
            plane.rotation.x = -Math.PI / 2;
            plane.position.set(x, 0.008, z);
            scene.add(plane);
        }

        function createRollUpShutterDoor(x, z, titleText) {
            const doorGroup = new THREE.Group();

            const frameMat = new THREE.MeshStandardMaterial({ color: 0xdc2626, metalness: 0.7 });
            const leftPillar = new THREE.Mesh(new THREE.BoxGeometry(0.3, 4.2, 0.3), frameMat);
            leftPillar.position.set(x - 2.0, 2.1, z); doorGroup.add(leftPillar);

            const rightPillar = new THREE.Mesh(new THREE.BoxGeometry(0.3, 4.2, 0.3), frameMat);
            rightPillar.position.set(x + 2.0, 2.1, z); doorGroup.add(rightPillar);

            const boxMat = new THREE.MeshStandardMaterial({ color: 0x1e293b, metalness: 0.8 });
            const topBox = new THREE.Mesh(new THREE.BoxGeometry(4.4, 0.6, 0.5), boxMat);
            topBox.position.set(x, 4.2, z); doorGroup.add(topBox);

            const slatMat = new THREE.MeshStandardMaterial({ color: 0xcbd5e1, metalness: 0.8, roughness: 0.3 });
            for (let i = 0; i < 12; i++) {
                const slat = new THREE.Mesh(new THREE.BoxGeometry(3.8, 0.28, 0.08), slatMat);
                slat.position.set(x, 0.25 + (i * 0.31), z);
                doorGroup.add(slat);
            }

            scene.add(doorGroup);
        }

        function createAluminumGlassExitDoor(x, z) {
            const doorGroup = new THREE.Group();

            const aluMat = new THREE.MeshStandardMaterial({ color: 0x94a3b8, metalness: 0.9, roughness: 0.2 });
            const outerFrame = new THREE.Mesh(new THREE.BoxGeometry(0.15, 3.0, 1.8), aluMat);
            outerFrame.position.set(x, 1.5, z); doorGroup.add(outerFrame);

            const glassMat = new THREE.MeshStandardMaterial({ color: 0x38bdf8, transparent: true, opacity: 0.35, roughness: 0.1 });
            const glassPane = new THREE.Mesh(new THREE.BoxGeometry(0.08, 2.7, 1.5), glassMat);
            glassPane.position.set(x, 1.4, z); doorGroup.add(glassPane);

            const handleMat = new THREE.MeshStandardMaterial({ color: 0xf8fafc, metalness: 0.95 });
            const handle = new THREE.Mesh(new THREE.BoxGeometry(0.2, 0.5, 0.06), handleMat);
            handle.position.set(x + 0.1, 1.4, z - 0.5); doorGroup.add(handle);

            scene.add(doorGroup);
        }

        function createRoofTrussesAndDecorations() {
            const trussMat = new THREE.MeshStandardMaterial({ color: 0xdc2626, metalness: 0.6 });
            for (let z = 4; z < WORLD_SIZE; z += 6) {
                const truss = new THREE.Mesh(new THREE.BoxGeometry(WORLD_SIZE, 0.2, 0.2), trussMat);
                truss.position.set(WORLD_SIZE / 2, 7.8, z);
                scene.add(truss);
            }

            const extMat = new THREE.MeshStandardMaterial({ color: 0xef4444, roughness: 0.3 });
            const ext = new THREE.Mesh(new THREE.CylinderGeometry(0.12, 0.12, 0.6, 16), extMat);
            ext.position.set(0.25, 1.2, 10.0);
            scene.add(ext);
        }

        function createFPSPlayerHands() {
            fpsArmGroup = new THREE.Group();

            const armGeo = new THREE.CylinderGeometry(0.055, 0.065, 0.45, 16);
            const sleeveMat = new THREE.MeshStandardMaterial({ color: 0xdc2626 });
            const arm = new THREE.Mesh(armGeo, sleeveMat);
            arm.rotation.x = Math.PI / 3; arm.rotation.z = -Math.PI / 8;
            arm.position.set(0.22, -0.22, -0.38);
            fpsArmGroup.add(arm);

            const gloveMat = new THREE.MeshStandardMaterial({ color: 0x475569 });
            const hand = new THREE.Mesh(new THREE.BoxGeometry(0.09, 0.08, 0.12), gloveMat);
            hand.position.set(0.26, -0.16, -0.52);
            fpsArmGroup.add(hand);

            const pdaMat = new THREE.MeshStandardMaterial({ color: 0x0f172a, metalness: 0.8 });
            const pda = new THREE.Mesh(new THREE.BoxGeometry(0.09, 0.18, 0.04), pdaMat);
            pda.rotation.x = Math.PI / 4;
            pda.position.set(0.26, -0.12, -0.55);
            fpsArmGroup.add(pda);

            const screenMat = new THREE.MeshBasicMaterial({ color: 0xef4444 });
            const pdaScreen = new THREE.Mesh(new THREE.PlaneGeometry(0.07, 0.11), screenMat);
            pdaScreen.rotation.x = -Math.PI / 4;
            pdaScreen.position.set(0.26, -0.10, -0.54);
            fpsArmGroup.add(pdaScreen);

            camera.add(fpsArmGroup);
            scene.add(camera);
        }

        function buildContinuousRacksWithFEFOAndEmptySlots() {
            let counts = { red: 0, orange: 0, blue: 0, green: 0 };

            const rackConfigs = [];
            // Staging racks rotated to vertical orientation.
            // Inbound Staging Rack removed: keep the inbound passage clear for AGVs.
            rackConfigs.push({ rowIdx: 2, startX: 24.0, startZ: 18.8, bays: 5, bayWidth: 1.0, depth: 0.8, levels: 3, title: "Despatch Staging Rack", orientation: "vertical" });

            let rIdx = 3;
            for (let rowZ = 4.0; rowZ <= 18.0; rowZ += 2.6) {
                rackConfigs.push({ rowIdx: rIdx++, startX: 3.0, startZ: rowZ, bays: 8, bayWidth: 1.1, depth: 0.9, levels: 10, title: "Main Storage Row A", orientation: "horizontal" });
                rackConfigs.push({ rowIdx: rIdx++, startX: 15.0, startZ: rowZ, bays: 8, bayWidth: 1.1, depth: 0.9, levels: 10, title: "Main Storage Row B", orientation: "horizontal" });
            }

            const physicalSlots = [];

            rackConfigs.forEach(function(config) {
                const startX = config.startX;
                const startZ = config.startZ;
                const bays = config.bays;
                const bayWidth = config.bayWidth;
                const depth = config.depth;
                const levels = config.levels;
                const title = config.title;
                const rowIdx = config.rowIdx;
                const orientation = config.orientation || "horizontal";

                const totalLength = bays * bayWidth;
                const rackHeight = levels * 0.5;
                const spanX = orientation === 'horizontal' ? totalLength : depth;
                const spanZ = orientation === 'horizontal' ? depth : totalLength;
                const centerX = orientation === 'horizontal' ? startX + totalLength / 2 : startX;
                const centerZ = orientation === 'horizontal' ? startZ : startZ + totalLength / 2;

                const bbox = new THREE.Box3(
                    new THREE.Vector3(centerX - spanX / 2 - 0.1, 0, centerZ - spanZ / 2 - 0.1),
                    new THREE.Vector3(centerX + spanX / 2 + 0.1, rackHeight + 0.2, centerZ + spanZ / 2 + 0.1)
                );
                obstacleBoxes.push(bbox);
                markGridUnwalkable(centerX - spanX / 2, centerZ, spanX, spanZ);

                const pillarMat = new THREE.MeshStandardMaterial({ color: 0xdc2626, metalness: 0.7 });
                const beamMat = new THREE.MeshStandardMaterial({ color: 0xf8fafc, metalness: 0.3 });

                for (let b = 0; b <= bays; b++) {
                    if (orientation === 'horizontal') {
                        const px = startX + (b * bayWidth);
                        const p1 = new THREE.Mesh(new THREE.BoxGeometry(0.08, rackHeight, 0.08), pillarMat);
                        p1.position.set(px, rackHeight / 2, startZ - depth / 2); scene.add(p1);
                        const p2 = new THREE.Mesh(new THREE.BoxGeometry(0.08, rackHeight, 0.08), pillarMat);
                        p2.position.set(px, rackHeight / 2, startZ + depth / 2); scene.add(p2);
                    } else {
                        const pz = startZ + (b * bayWidth);
                        const p1 = new THREE.Mesh(new THREE.BoxGeometry(0.08, rackHeight, 0.08), pillarMat);
                        p1.position.set(startX - depth / 2, rackHeight / 2, pz); scene.add(p1);
                        const p2 = new THREE.Mesh(new THREE.BoxGeometry(0.08, rackHeight, 0.08), pillarMat);
                        p2.position.set(startX + depth / 2, rackHeight / 2, pz); scene.add(p2);
                    }
                }

                for (let lvl = 1; lvl <= levels; lvl++) {
                    const beamY = lvl * 0.5;
                    if (orientation === 'horizontal') {
                        const b1 = new THREE.Mesh(new THREE.BoxGeometry(totalLength, 0.05, 0.05), beamMat);
                        b1.position.set(startX + totalLength / 2, beamY, startZ - depth / 2); scene.add(b1);
                        const b2 = new THREE.Mesh(new THREE.BoxGeometry(totalLength, 0.05, 0.05), beamMat);
                        b2.position.set(startX + totalLength / 2, beamY, startZ + depth / 2); scene.add(b2);
                    } else {
                        const b1 = new THREE.Mesh(new THREE.BoxGeometry(0.05, 0.05, totalLength), beamMat);
                        b1.position.set(startX - depth / 2, beamY, startZ + totalLength / 2); scene.add(b1);
                        const b2 = new THREE.Mesh(new THREE.BoxGeometry(0.05, 0.05, totalLength), beamMat);
                        b2.position.set(startX + depth / 2, beamY, startZ + totalLength / 2); scene.add(b2);
                    }
                }

                for (let b = 0; b < bays; b++) {
                    const slotX = orientation === 'horizontal' ? startX + (b * bayWidth) + (bayWidth / 2) : startX;
                    const slotZ = orientation === 'horizontal' ? startZ : startZ + (b * bayWidth) + (bayWidth / 2);
                    for (let lvl = 1; lvl <= levels; lvl++) {
                        const slotY = lvl * 0.5;
                        const distToInbound = Math.hypot(slotX - INBOUND_DOOR_POS.x, slotZ - INBOUND_DOOR_POS.z);
                        const distToDespatch = Math.hypot(slotX - DESPATCH_DOOR_POS.x, slotZ - DESPATCH_DOOR_POS.z);
                        const minDockDist = Math.min(distToInbound, distToDespatch);
                        const rowStr = rowIdx < 10 ? '0' + rowIdx : rowIdx;
                        const lvlStr = lvl < 10 ? '0' + lvl : lvl;
                        const locCode = `H${rowStr}T${lvlStr}`;

                        physicalSlots.push({
                            x: slotX, y: slotY, z: slotZ,
                            bayWidth: bayWidth, depth: depth,
                            rackTitle: title, level: lvl, maxLevels: levels,
                            rowIdx: rowIdx, locationCode: locCode, minDockDist: minDockDist
                        });
                    }
                }
            });

            physicalSlots.sort(function(a, b) { return a.minDockDist - b.minDockDist; });

            // Balance actual Kaggle goods across the three FEFO classes.
            // Only allocate one real row per shelf slot; never duplicate or invent goods.
            // Keep the physically empty slots at their existing positions.
            const capacity = physicalSlots.reduce((count, _, i) => count + (i % 7 !== 0 ? 1 : 0), 0);
            const buckets = [
                RAW_INVENTORY.filter(p => p.daysLeft >= 0 && p.daysLeft <= 15 && p.quantity > 0),
                RAW_INVENTORY.filter(p => p.daysLeft >= 16 && p.daysLeft <= 45 && p.quantity > 0),
                RAW_INVENTORY.filter(p => p.daysLeft > 45 && p.quantity > 0)
            ];
            buckets.forEach(b => b.sort((a,b)=>a.daysLeft-b.daysLeft));
            const offsets=[0,0,0], selectedInventory=[];
            // Round-robin quotas distribute capacity fairly; scarcity in one bucket
            // automatically transfers shelf capacity to other real product records.
            while(selectedInventory.length<capacity){
                let added=false;
                for(let group=0;group<3 && selectedInventory.length<capacity;group++){
                    if(offsets[group]<buckets[group].length){
                        selectedInventory.push(buckets[group][offsets[group]++]);
                        added=true;
                    }
                }
                if(!added)break;
            }
            // Physical FEFO: urgent goods sit closest to dispatch/inbound doors.
            selectedInventory.sort((a,b)=>a.daysLeft-b.daysLeft);
            const selectedCounts=[
                selectedInventory.filter(p=>p.daysLeft<=15).length,
                selectedInventory.filter(p=>p.daysLeft>=16 && p.daysLeft<=45).length,
                selectedInventory.filter(p=>p.daysLeft>45).length
            ];
            console.info(`FEFO stock from Kaggle • Red: ${selectedCounts[0]}, Yellow: ${selectedCounts[1]}, Green: ${selectedCounts[2]}`);
            const summary=document.getElementById('fefo-stock-diversity');
            if(summary)summary.textContent=`🔴 ${selectedCounts[0]} gấp · 🟡 ${selectedCounts[1]} trung bình · 🟢 ${selectedCounts[2]} không gấp`;
            let invIndex = 0;
            for (let i = 0; i < physicalSlots.length; i++) {
                const slotLoc = physicalSlots[i];
                const isEmptySlot = (i % 7 === 0 || invIndex >= selectedInventory.length);

                let slotData = null;

                if (!isEmptySlot && invIndex < selectedInventory.length) {
                    const item = selectedInventory[invIndex++];

                    // FEFO colours: RED = urgent (<=15 d), YELLOW = watch (16–45 d),
                    // GREEN = longest remaining life (>45 d). Driven only by Kaggle expiry dates.
                    let statusStr, colorGroup, colorHex;
                    if (item.daysLeft <= 15) {
                        colorHex=0xef4444; statusStr="🔴 ĐỎ - CẦN XUẤT TRƯỚC (≤15 NGÀY)";
                        colorGroup='red'; counts.red++;
                    } else if (item.daysLeft <= 45) {
                        colorHex=0xfacc15; statusStr="🟡 VÀNG - SẮP ĐẾN HẠN (16–45 NGÀY)";
                        colorGroup='orange'; counts.orange++;
                    } else {
                        colorHex=0x22c55e; statusStr="🟢 XANH - HẠN CÒN DÀI (>45 NGÀY)";
                        colorGroup='green'; counts.green++;
                    }
                    const statusColorHex=colorHex;

                    slotData = Object.assign({}, item, {
                        id: i, isEmpty: false, locationCode: slotLoc.locationCode,
                        rackTitle: slotLoc.rackTitle, level: slotLoc.level,
                        x: slotLoc.x, y: slotLoc.y, z: slotLoc.z,
                        colorGroup: colorGroup, colorHex: colorHex, statusColorHex: statusColorHex, statusStr: statusStr
                    });

                    const boxGeo = new THREE.BoxGeometry(slotLoc.bayWidth * 0.75, 0.38, slotLoc.depth * 0.75);
                    const boxMat = new THREE.MeshStandardMaterial({ color: colorHex, roughness: 0.35, metalness: 0.02, emissive: colorHex, emissiveIntensity: 0.10 });
                    const boxMesh = new THREE.Mesh(boxGeo, boxMat);
                    boxMesh.position.set(slotLoc.x, slotLoc.y, slotLoc.z);
                    boxMesh.castShadow = true;
                    boxMesh.userData = slotData;

                    scene.add(boxMesh);
                    slotMeshes.push(boxMesh);
                } else {
                    slotData = {
                        id: i, isEmpty: true, productId: `EMPTY-${slotLoc.locationCode}`, sku: "EMPTY",
                        productName: "🔲 Ô KỆ TRỐNG (EMPTY SLOT)", category: "Empty", quantity: 0, daysLeft: 999,
                        locationCode: slotLoc.locationCode, rackTitle: slotLoc.rackTitle, level: slotLoc.level,
                        x: slotLoc.x, y: slotLoc.y, z: slotLoc.z, colorHex: 0x94a3b8, statusStr: "⚪ Ô TRỐNG SẴN SÀNG"
                    };

                    const ghostGeo = new THREE.BoxGeometry(slotLoc.bayWidth * 0.75, 0.38, slotLoc.depth * 0.75);
                    const ghostMat = new THREE.MeshStandardMaterial({ color: 0xcbd5e1, wireframe: true, transparent: true, opacity: 0.4 });
                    const ghostMesh = new THREE.Mesh(ghostGeo, ghostMat);
                    ghostMesh.position.set(slotLoc.x, slotLoc.y, slotLoc.z);
                    ghostMesh.userData = slotData;

                    scene.add(ghostMesh);
                    slotMeshes.push(ghostMesh);
                }

                slotsData.push(slotData);
            }

            document.getElementById('tab-sum-red').innerText = counts.red;
            document.getElementById('tab-sum-orange').innerText = counts.orange;
            // Three FEFO levels only: no blue group.
            document.getElementById('tab-sum-green').innerText = counts.green;
        }

        function createChargingStation3D() {
            chargingStationMesh = new THREE.Group();

            const padGeo = new THREE.BoxGeometry(1.95, 0.08, 8.4);
            const padMat = new THREE.MeshStandardMaterial({ color: 0xdc2626, metalness: 0.8, roughness: 0.2 });
            const pad = new THREE.Mesh(padGeo, padMat);
            pad.position.set(CHARGING_STATION_POS.x, 0.04, CHARGING_STATION_POS.z);
            chargingStationMesh.add(pad);

            const postGeo = new THREE.BoxGeometry(0.4, 2.2, 0.4);
            const postMat = new THREE.MeshStandardMaterial({ color: 0x1e293b, metalness: 0.8 });
            const post = new THREE.Mesh(postGeo, postMat);
            post.position.set(CHARGING_STATION_POS.x - 0.72, 1.1, CHARGING_STATION_POS.z - 3.75);
            chargingStationMesh.add(post);

            const ledGeo = new THREE.SphereGeometry(0.18, 16, 16);
            const ledMat = new THREE.MeshBasicMaterial({ color: 0xef4444 });
            const led = new THREE.Mesh(ledGeo, ledMat);
            led.position.set(CHARGING_STATION_POS.x - 0.72, 2.1, CHARGING_STATION_POS.z - 3.75);
            chargingStationMesh.add(led);

            // Mark each of the five individual charging spots with an emissive disc.
            CHARGING_BAYS.forEach((bay, index) => {
                const marker = new THREE.Mesh(
                    new THREE.CylinderGeometry(0.33,0.33,0.015,24),
                    new THREE.MeshBasicMaterial({color:0xfef08a,transparent:true,opacity:0.65}));
                marker.position.set(bay.x,0.09,bay.z);
                chargingStationMesh.add(marker);
            });
            scene.add(chargingStationMesh);
        }

        function startTabletClock() {
            setInterval(() => {
                const now = new Date();
                const clockEl = document.getElementById('tablet-clock');
                if (clockEl) clockEl.innerText = now.toLocaleTimeString([], { hour: '2-digit', minute: '2-digit' });
            }, 1000);
        }

        function populateTabletDataAndOptions() {
            const selectItemEl = document.getElementById('agv-select-item');
            const selectDropEl = document.getElementById('agv-select-drop');

            selectItemEl.innerHTML = '';

            const occupiedSlots = slotsData.filter(s => !s.isEmpty);
            occupiedSlots.forEach(s => {
                const opt = document.createElement('option');
                opt.value = s.id;
                opt.innerText = `[${s.locationCode}] SKU: ${s.sku} - ${s.productName}`;
                selectItemEl.appendChild(opt);
            });

            slotsData.forEach(s => {
                const opt = document.createElement('option');
                opt.value = s.locationCode;
                opt.innerText = s.isEmpty ?
                    `🔲 Mã [${s.locationCode}] - Ô Trống (Cất Hàng)` :
                    `📦 Mã [${s.locationCode}] - ${s.productName}`;
                selectDropEl.appendChild(opt);
            });
        }

        function filterAGVProductList() {
            const q = document.getElementById('agv-sku-input').value.toLowerCase().trim();
            const selectItemEl = document.getElementById('agv-select-item');
            selectItemEl.innerHTML = '';

            const filtered = slotsData.filter(s => !s.isEmpty && (
                s.sku.toLowerCase().includes(q) ||
                s.productId.toLowerCase().includes(q) ||
                s.productName.toLowerCase().includes(q) ||
                s.locationCode.toLowerCase().includes(q)
            ));

            filtered.forEach(s => {
                const opt = document.createElement('option');
                opt.value = s.id;
                opt.innerText = `[${s.locationCode}] SKU: ${s.sku} - ${s.productName}`;
                selectItemEl.appendChild(opt);
            });
        }

        function setupMiniMapCanvas() {
            const canvas = document.getElementById('minimap-canvas');
            if (!canvas) return;
            const ctx = canvas.getContext('2d');

            function drawMiniMap() {
                ctx.clearRect(0, 0, canvas.width, canvas.height);

                ctx.fillStyle = '#0f172a';
                ctx.fillRect(0, 0, canvas.width, canvas.height);

                const scaleX = canvas.width / WORLD_SIZE;
                const scaleY = canvas.height / WORLD_SIZE;

                ctx.fillStyle = '#dc2626';
                ctx.fillRect((CHARGING_STATION_POS.x - .975) * scaleX, (CHARGING_STATION_POS.z - 4.2) * scaleY, 1.95 * scaleX, 8.4 * scaleY);
                ctx.fillStyle = '#ffffff';
                ctx.font = 'bold 10px sans-serif';
                ctx.fillText('⚡SẠC', (CHARGING_STATION_POS.x - .75) * scaleX, (CHARGING_STATION_POS.z - 3.2) * scaleY);

                ctx.fillStyle = '#ef4444';
                ctx.fillRect((INBOUND_DOOR_POS.x - 2) * scaleX, (INBOUND_DOOR_POS.z - 0.5) * scaleY, 4 * scaleX, 1 * scaleY);
                ctx.fillStyle = '#ffffff';
                ctx.font = '9px sans-serif';
                ctx.fillText('INBOUND', (INBOUND_DOOR_POS.x - 1.5) * scaleX, (INBOUND_DOOR_POS.z + 0.2) * scaleY);

                ctx.fillStyle = '#10b981';
                ctx.fillRect((DESPATCH_DOOR_POS.x - 2) * scaleX, (DESPATCH_DOOR_POS.z - 0.5) * scaleY, 4 * scaleX, 1 * scaleY);
                ctx.fillStyle = '#ffffff';
                ctx.fillText('DESPATCH', (DESPATCH_DOOR_POS.x - 1.5) * scaleX, (DESPATCH_DOOR_POS.z + 0.2) * scaleY);

                slotsData.forEach(s => {
                    const cx = s.x * scaleX;
                    const cy = s.z * scaleY;

                    if (s.isEmpty) {
                        ctx.fillStyle = '#475569';
                        ctx.fillRect(cx - 2, cy - 2, 4, 4);
                    } else {
                        ctx.fillStyle = s.colorHex ? '#' + s.colorHex.toString(16).padStart(6, '0') : '#ef4444';
                        ctx.fillRect(cx - 2, cy - 2, 4, 4);
                    }
                });

                 // Draw live routes of the five AGVs in their corresponding colours.
                 fleet5.forEach(r=>{
                     if(r.path.length>1 && r.index<r.path.length){
                         ctx.beginPath();
                         ctx.moveTo(r.mesh.position.x*scaleX,r.mesh.position.z*scaleY);
                         r.path.slice(r.index).forEach(p=>ctx.lineTo(p.x*scaleX,p.z*scaleY));
                         ctx.strokeStyle='#'+r.color.toString(16).padStart(6,'0');
                         ctx.lineWidth=2.1;ctx.setLineDash([5,3]);ctx.stroke();ctx.setLineDash([]);
                     }
                 });

                 fleet5.forEach(r=>{
                     ctx.fillStyle='#'+r.color.toString(16).padStart(6,'0');
                     const x=r.mesh.position.x*scaleX,z=r.mesh.position.z*scaleY;
                     ctx.beginPath();ctx.arc(x,z,5,0,Math.PI*2);ctx.fill();
                     ctx.strokeStyle='#fff';ctx.lineWidth=1;ctx.stroke();
                     ctx.font='bold 9px Arial';ctx.fillStyle='#fff';ctx.fillText(String(r.id+1),x+6,z-5);
                 });
                requestAnimationFrame(drawMiniMap);
            }

            drawMiniMap();

            canvas.addEventListener('click', function(e) {
                const rect = canvas.getBoundingClientRect();
                const clickX = (e.clientX - rect.left) * (WORLD_SIZE / canvas.width);
                const clickZ = (e.clientY - rect.top) * (WORLD_SIZE / canvas.height);

                let closestSlot = null;
                let minDist = 999;

                slotsData.forEach(s => {
                    const d = Math.hypot(s.x - clickX, s.z - clickZ);
                    if (d < minDist) {
                        minDist = d;
                        closestSlot = s;
                    }
                });

                if (closestSlot && minDist < 3.0) {
                    const dropSelect = document.getElementById('agv-select-drop');
                    dropSelect.value = closestSlot.locationCode;
                    document.getElementById('minimap-selected-loc').innerText = closestSlot.locationCode;
                }
            });
        }

        function syncDropLocationFromSelect() {
            const val = document.getElementById('agv-select-drop').value;
            document.getElementById('minimap-selected-loc').innerText = val;
        }

        function syncSelectedSlotToMap() {
            const slotId = document.getElementById('agv-select-item').value;
            const slot = slotsData.find(s => s.id == slotId);
            if (slot) {
                document.getElementById('minimap-selected-loc').innerText = slot.locationCode;
            }
        }

        function markGridUnwalkable(startX, startZ, length, depth) {
            const minX = Math.max(0, Math.floor((startX - 0.25) / GRID_RES));
            const maxX = Math.min(GRID_DIM - 1, Math.floor((startX + length + 0.25) / GRID_RES));
            const minZ = Math.max(0, Math.floor((startZ - depth/2 - 0.25) / GRID_RES));
            const maxZ = Math.min(GRID_DIM - 1, Math.floor((startZ + depth/2 + 0.25) / GRID_RES));

            for (let x = minX; x <= maxX; x++) {
                for (let z = minZ; z <= maxZ; z++) {
                    gridMap[x][z].walkable = false;
                }
            }
        }

        function getNearestWalkableNode(worldX, worldZ) {
            let gx = Math.floor(worldX / GRID_RES);
            let gz = Math.floor(worldZ / GRID_RES);

            gx = Math.max(0, Math.min(GRID_DIM - 1, gx));
            gz = Math.max(0, Math.min(GRID_DIM - 1, gz));

            if (gridMap[gx] && gridMap[gx][gz] && gridMap[gx][gz].walkable) {
                return { x: gx, z: gz };
            }

            for (let r = 1; r <= 8; r++) {
                for (let dx = -r; dx <= r; dx++) {
                    for (let dz = -r; dz <= r; dz++) {
                        if (Math.abs(dx) === r || Math.abs(dz) === r) {
                            let nx = gx + dx;
                            let nz = gz + dz;
                            if (nx >= 0 && nx < GRID_DIM && nz >= 0 && nz < GRID_DIM) {
                                if (gridMap[nx][nz].walkable) {
                                    return { x: nx, z: nz };
                                }
                            }
                        }
                    }
                }
            }
            return { x: gx, z: gz };
        }

        // A* route preview: four-neighbour grid, Manhattan heuristic, binary min-heap.
        // The live fleet planner above uses the same cost model with soft robot penalties.
        function findPathAStar(startPos, targetPos) {
            const startNode = getNearestWalkableNode(startPos.x, startPos.z);
            const targetNode = getNearestWalkableNode(targetPos.x, targetPos.z);
            const sx=startNode.x, sz=startNode.z, tx=targetNode.x, tz=targetNode.z;
            const key=(x,z)=>x*GRID_DIM+z;
            const sourceKey=key(sx,sz), targetKey=key(tx,tz);
            const total=GRID_DIM*GRID_DIM;
            const gScore=new Float64Array(total); gScore.fill(Infinity);
            const parent=new Int32Array(total); parent.fill(-1);
            const closed=new Uint8Array(total);
            const open=[];
            const heuristic=(x,z)=>Math.abs(x-tx)+Math.abs(z-tz);
            const push=item=>{
                let i=open.length; open.push(item);
                while(i){
                    const p=(i-1)>>1;
                    if(open[p].f<=item.f)break;
                    open[i]=open[p]; i=p;
                }
                open[i]=item;
            };
            const pop=()=>{
                const head=open[0], last=open.pop();
                if(open.length){
                    let i=0;
                    while((i*2+1)<open.length){
                        let child=i*2+1, right=child+1;
                        if(right<open.length && open[right].f<open[child].f)child=right;
                        if(open[child].f>=last.f)break;
                        open[i]=open[child]; i=child;
                    }
                    open[i]=last;
                }
                return head;
            };
            gScore[sourceKey]=0; push({x:sx,z:sz,f:heuristic(sx,sz)});
            const dirs=[[1,0],[-1,0],[0,1],[0,-1]];
            while(open.length){
                const current=pop(), ck=key(current.x,current.z);
                if(closed[ck])continue;
                closed[ck]=1;
                if(ck===targetKey)break;
                for(const [dx,dz] of dirs){
                    const nx=current.x+dx, nz=current.z+dz;
                    if(nx<0||nz<0||nx>=GRID_DIM||nz>=GRID_DIM||!gridMap[nx][nz].walkable)continue;
                    const nk=key(nx,nz);
                    if(closed[nk])continue;
                    const tentative=gScore[ck]+1;
                    if(tentative<gScore[nk]){
                        gScore[nk]=tentative; parent[nk]=ck;
                        push({x:nx,z:nz,f:tentative+heuristic(nx,nz)});
                    }
                }
            }
            if(!closed[targetKey])return [];
            const path=[];
            for(let k=targetKey;k!==-1;k=parent[k]){
                const x=Math.floor(k/GRID_DIM), z=k%GRID_DIM;
                path.push(new THREE.Vector3(x*GRID_RES+0.25,0.15,z*GRID_RES+0.25));
            }
            return path.reverse();
        }
        function renderPath3D(pathPoints, targetSlotData) {
            if (pathLineMesh) scene.remove(pathLineMesh);

            if (!pathPoints || pathPoints.length < 1) return;

            const waypoints = [];
            for (let i = 0; i < pathPoints.length; i++) {
                waypoints.push(pathPoints[i].clone());
            }

            const lastGridPt = waypoints[waypoints.length - 1].clone();

            if (Math.abs(lastGridPt.x - targetSlotData.x) > 0.02) {
                waypoints.push(new THREE.Vector3(targetSlotData.x, 0.15, lastGridPt.z));
            }
            if (Math.abs(lastGridPt.z - targetSlotData.z) > 0.02) {
                waypoints.push(new THREE.Vector3(targetSlotData.x, 0.15, targetSlotData.z));
            }
            if (Math.abs(0.15 - targetSlotData.y) > 0.02) {
                waypoints.push(new THREE.Vector3(targetSlotData.x, targetSlotData.y, targetSlotData.z));
            }

            const cleanWaypoints = [];
            for (let i = 0; i < waypoints.length; i++) {
                if (i === 0 || waypoints[i].distanceTo(cleanWaypoints[cleanWaypoints.length - 1]) > 0.02) {
                    cleanWaypoints.push(waypoints[i]);
                }
            }

            if (cleanWaypoints.length < 2) return;

            const curvePath = new THREE.CurvePath();
            for (let i = 0; i < cleanWaypoints.length - 1; i++) {
                curvePath.add(new THREE.LineCurve3(cleanWaypoints[i], cleanWaypoints[i + 1]));
            }

            const tubeGeo = new THREE.TubeGeometry(curvePath, cleanWaypoints.length * 6, 0.09, 8, false);
            const tubeMat = new THREE.MeshBasicMaterial({ color: 0xef4444 });

            pathLineMesh = new THREE.Mesh(tubeGeo, tubeMat);
            scene.add(pathLineMesh);

            const totalDistMeter = (cleanWaypoints.length * 0.5).toFixed(1);
            const levelStr = targetSlotData.level < 10 ? '0' + targetSlotData.level : targetSlotData.level;

            document.getElementById('route-result-box').classList.remove('hidden');
            document.getElementById('tab-res-title').innerText = targetSlotData.productName;
            document.getElementById('tab-res-sku').innerText = targetSlotData.sku;
            document.getElementById('tab-res-loc').innerText = targetSlotData.locationCode;
            document.getElementById('tab-res-days').innerText = targetSlotData.daysLeft + ' Ngày';
            document.getElementById('tab-res-qty').innerText = targetSlotData.quantity + ' đơn vị';
            document.getElementById('tab-res-dist').innerText = totalDistMeter + ' m';
            document.getElementById('tab-res-status').innerText = targetSlotData.statusStr;

            document.getElementById('tablet-modal').classList.add('hidden');
        }

        function dispatchInspectedFleetItem(){
            if(currentInspectedSlot&&!currentInspectedSlot.isEmpty)
                f5Assign(fleetRobotByChoice(),currentInspectedSlot,'DESPATCH');
        }
        function relocateInspectedItem() {
            if (!currentInspectedSlot) return;

            const emptySlots = slotsData.filter(s => s.isEmpty);
            if (emptySlots.length === 0) {
                alert("Không còn ô trống trong kho!");
                return;
            }

            const targetCode = prompt(
                `🔄 BẠN ĐANG DỜI SẢN PHẨM: [${currentInspectedSlot.sku}] ${currentInspectedSlot.productName}\\n\\n` +
                `Nhập Mã Kệ Muốn Dời Tới (Ví dụ: ${emptySlots[0].locationCode}):`,
                emptySlots[0].locationCode
            );

            if (targetCode) {
                const targetSlot = slotsData.find(s => s.locationCode.toUpperCase() === targetCode.toUpperCase().trim());
                if (targetSlot) {
                    f5Assign(fleetRobotByChoice(),currentInspectedSlot,targetSlot.locationCode);
                } else {
                    alert("Mã Kệ không tồn tại!");
                }
            }
        }

        function setupEventListeners() {
            window.addEventListener('resize', onWindowResize);
            document.addEventListener('fullscreenchange', onWindowResize);

            const container = document.getElementById('canvas-container');
            const overlay = document.getElementById('instructions-overlay');

            container.addEventListener('click', function(e) {
                if (e.target.closest('#tablet-modal') || e.target.closest('button') || e.target.closest('#inspector-panel') || e.target.closest('input') || e.target.closest('select') || e.target.closest('canvas')) return;

                overlay.classList.add('hidden');

                if (isFPSMode && !document.pointerLockElement) {
                    try { document.body.requestPointerLock(); } catch (err) {}
                }
            });

            document.addEventListener('keydown', function(e) {
                if (e.key === 'Alt' || e.code === 'AltLeft' || e.code === 'AltRight') {
                    e.preventDefault();
                    if (document.pointerLockElement) {
                        document.exitPointerLock();
                    }
                    isMouseDown = false;
                    return;
                }

                switch (e.code) {
                    case 'KeyW': moveForward = true; break;
                    case 'KeyS': moveBackward = true; break;
                    case 'KeyA': moveLeft = true; break;
                    case 'KeyD': moveRight = true; break;
                    case 'ShiftLeft': isSprinting = true; break;
                }
            });

            document.addEventListener('keyup', function(e) {
                if (e.key === 'Alt' || e.code === 'AltLeft' || e.code === 'AltRight') {
                    e.preventDefault();
                    return;
                }

                switch (e.code) {
                    case 'KeyW': moveForward = false; break;
                    case 'KeyS': moveBackward = false; break;
                    case 'KeyA': moveLeft = false; break;
                    case 'KeyD': moveRight = false; break;
                    case 'ShiftLeft': isSprinting = false; break;
                }
            });

            document.addEventListener('mousedown', function(e) {
                if (e.target.closest('#tablet-modal') || e.target.closest('button') || e.target.closest('#inspector-panel') || e.target.closest('input') || e.target.closest('select')) return;
                if (e.button === 0 && isFPSMode) {
                    isMouseDown = true;
                    prevMouseX = e.clientX;
                    prevMouseY = e.clientY;
                }
            });

            document.addEventListener('mouseup', function() { isMouseDown = false; });

            document.addEventListener('mousemove', function(e) {
                if (!isFPSMode) return;

                if (document.pointerLockElement) {
                    yaw -= e.movementX * 0.0022;
                    pitch -= e.movementY * 0.0022;
                    pitch = Math.max(-Math.PI / 2.2, Math.min(Math.PI / 2.2, pitch));
                } else if (isMouseDown) {
                    const deltaX = e.clientX - prevMouseX;
                    const deltaY = e.clientY - prevMouseY;
                    prevMouseX = e.clientX;
                    prevMouseY = e.clientY;

                    yaw -= deltaX * 0.0025;
                    pitch -= deltaY * 0.0025;
                    pitch = Math.max(-Math.PI / 2.2, Math.min(Math.PI / 2.2, pitch));
                }
            });

            window.addEventListener('click', function(e) {
                if (e.target.closest('#tablet-modal') || e.target.closest('button') || e.target.closest('#inspector-panel')) return;
                inspectCurrentTarget();
            });
        }

        function inspectCurrentTarget() {
            if (!isFPSMode) return;
            const raycaster = new THREE.Raycaster();
            raycaster.setFromCamera(new THREE.Vector2(0, 0), camera);
            const intersects = raycaster.intersectObjects(slotMeshes);

            if (intersects.length > 0) {
                const data = intersects[0].object.userData;
                showInspector(data, intersects[0].object);
            }
        }

        function showInspector(data, mesh) {
            currentInspectedSlot = data;
            const panel = document.getElementById('inspector-panel');
            panel.classList.remove('hidden');

            document.getElementById('inspect-title').innerText = data.productName;
            document.getElementById('inspect-loc').innerText = data.locationCode;
            document.getElementById('inspect-sku').innerText = data.sku || data.productId;
            document.getElementById('inspect-pos').innerText = `${data.rackTitle} - Tầng ${data.level}`;
            document.getElementById('inspect-qty').innerText = `${data.quantity} đơn vị (${data.category})`;
            document.getElementById('inspect-exp').innerText = data.isEmpty ? 'N/A' : `${data.expiryDateStr} (${data.daysLeft} ngày)`;
            document.getElementById('inspect-tag').innerText = data.statusStr;

            if (highlightMesh) scene.remove(highlightMesh);
            highlightMesh = new THREE.BoxHelper(mesh, 0xdc2626);
            scene.add(highlightMesh);
        }

        function closeInspector() {
            document.getElementById('inspector-panel').classList.add('hidden');
            if (highlightMesh) scene.remove(highlightMesh);
            currentInspectedSlot = null;
        }

        function toggleFullscreen() {
            const target = document.getElementById('canvas-container');
            if (!document.fullscreenElement) {
                target.requestFullscreen().catch(err => {});
            } else {
                if (document.exitFullscreen) document.exitFullscreen();
            }
            setTimeout(onWindowResize, 120);
        }

        function toggleTablet() {
            const tablet = document.getElementById('tablet-modal');
            tablet.classList.toggle('hidden');
            if (!tablet.classList.contains('hidden') && document.pointerLockElement) {
                document.exitPointerLock();
            }
        }

        function switchTab(tabId) {
            document.querySelectorAll('.tab-content').forEach(function(el) {
                el.classList.add('hidden'); el.classList.remove('flex');
            });
            document.querySelectorAll('.tab-btn').forEach(function(el) { el.classList.remove('active'); });
            const content=document.getElementById(tabId);
            content.classList.remove('hidden');content.classList.add('flex');
            document.getElementById('btn-' + tabId).classList.add('active');
            if(tabId==='tab-history')renderTaskHistory();
        }

        function searchAndNavigate() {
            const q = document.getElementById('tablet-search-input').value.toLowerCase().trim();
            if (!q) return;

            const target = slotsData.find(function(s) {
                const pidMatch = s.productId && s.productId.toLowerCase().includes(q);
                const skuMatch = s.sku && s.sku.toLowerCase().includes(q);
                const nameMatch = s.productName && s.productName.toLowerCase().includes(q);
                const locMatch = s.locationCode && s.locationCode.toLowerCase().includes(q);
                return pidMatch || skuMatch || nameMatch || locMatch;
            });

            if (target) {
                const path = findPathAStar(camera.position, new THREE.Vector3(target.x, 0, target.z));
                renderPath3D(path, target);
            } else {
                alert("Không tìm thấy mã SKU / Location này!");
            }
        }

        function navigateCriticalFEFO() {
            let occupied = slotsData.filter(s => !s.isEmpty).sort(function(a, b) { return a.daysLeft - b.daysLeft; });
            const target = occupied[0];
            if (target) {
                const path = findPathAStar(camera.position, new THREE.Vector3(target.x, 0, target.z));
                renderPath3D(path, target);
            }
        }

        function toggleViewMode() {
            isFPSMode = !isFPSMode;
            const modeText = document.getElementById('mode-text');
            const crosshair = document.getElementById('crosshair');

            if (isFPSMode) {
                modeText.innerText = "Góc Nhìn Nhân Viên (FPS)";
                crosshair.classList.remove('hidden');
                fpsArmGroup.visible = true;
                controls.enabled = false;
                camera.position.set(14.0, 1.6, 2.5);
                yaw = Math.PI;
                pitch = 0;
            } else {
                modeText.innerText = "Góc Nhìn Quản Lý (3D Orbit)";
                crosshair.classList.add('hidden');
                fpsArmGroup.visible = false;
                controls.enabled = true;
                camera.position.set(14, 25, 30);
                controls.target.set(14, 2, 14);
            }
        }

        function onWindowResize() {
            const container = document.getElementById('canvas-container');
            const w = container.clientWidth || window.innerWidth;
            const h = container.clientHeight || window.innerHeight;
            camera.aspect = w / h;
            camera.updateProjectionMatrix();
            renderer.setSize(w, h);
        }

        function animate() {
            requestAnimationFrame(animate);

            const time = performance.now();
            // Chặn delta-time quá lớn khi Colab vừa tải xong/tab bị treo.
            // Điều này ngăn bước nhảy vị trí đột ngột gây cảm giác giật/rung.
            const delta = Math.min(Math.max((time - prevTime) / 1000, 0), 0.05);
            prevTime = time;

            updateFiveFleet(delta);

            if (isFPSMode) {
                const euler = new THREE.Euler(pitch, yaw, 0, 'YXZ');
                camera.quaternion.setFromEuler(euler);

                const forward = new THREE.Vector3(0, 0, -1).applyAxisAngle(new THREE.Vector3(0, 1, 0), yaw);
                const right = new THREE.Vector3(1, 0, 0).applyAxisAngle(new THREE.Vector3(0, 1, 0), yaw);

                let moveVector = new THREE.Vector3(0, 0, 0);
                const speed = (isSprinting ? 10.0 : 4.5) * delta;

                if (moveForward) moveVector.addScaledVector(forward, speed);
                if (moveBackward) moveVector.addScaledVector(forward, -speed);
                if (moveRight) moveVector.addScaledVector(right, speed);
                if (moveLeft) moveVector.addScaledVector(right, -speed);

                const nextX = camera.position.x + moveVector.x;
                const playerBoxX = new THREE.Box3(
                    new THREE.Vector3(nextX - 0.25, 0.1, camera.position.z - 0.25),
                    new THREE.Vector3(nextX + 0.25, 1.8, camera.position.z + 0.25)
                );
                let collideX = false;
                for (let i = 0; i < obstacleBoxes.length; i++) {
                    if (playerBoxX.intersectsBox(obstacleBoxes[i])) { collideX = true; break; }
                }
                if (!collideX && nextX > 0.5 && nextX < WORLD_SIZE - 0.5) {
                    camera.position.x = nextX;
                }

                const nextZ = camera.position.z + moveVector.z;
                const playerBoxZ = new THREE.Box3(
                    new THREE.Vector3(camera.position.x - 0.25, 0.1, nextZ - 0.25),
                    new THREE.Vector3(camera.position.x + 0.25, 1.8, nextZ + 0.25)
                );
                let collideZ = false;
                for (let i = 0; i < obstacleBoxes.length; i++) {
                    if (playerBoxZ.intersectsBox(obstacleBoxes[i])) { collideZ = true; break; }
                }
                if (!collideZ && nextZ > 0.5 && nextZ < WORLD_SIZE - 0.5) {
                    camera.position.z = nextZ;
                }

                camera.position.y = 1.6;
            } else {
                controls.update();
            }

            renderer.render(scene, camera);
        }

        // Boot at DOM readiness; do not wait for unrelated CDN/font loads.
        if(document.readyState==='loading'){
            document.addEventListener('DOMContentLoaded',bootWarehouse,{once:true});
        } else {
            bootWarehouse();
        }
    </script>

</body>
</html>
"""

# Chèn dữ liệu JSON vào mẫu HTML
html_code = (html_template.replace("__INVENTORY_JSON_DATA__", inventory_json_str)
            .replace("__OPERATION_DATE__", today.strftime("%d/%m/%Y")))

# ------------------------------------------------------------------------------
# 3. LƯU VÀ HIỂN THỊ TRÊN GOOGLE COLAB
# ------------------------------------------------------------------------------
with open("warehouse_fefo_3d.html", "w", encoding="utf-8") as f:
    f.write(html_code)

print("✅ Đã xuất HTML 3D. Nếu Colab chặn WebGL/iframe, có thể mở file HTML riêng trên Chrome.")

html_b64 = base64.b64encode(html_code.encode('utf-8')).decode('utf-8')
# HTML output is generated with Python strings (no standalone HTML pasted into Python).
iframe_html = (
    '<div style="width:100%;text-align:center;margin-bottom:8px;">'
    '<p style="color:#dc2626;font-weight:bold;font-size:14px;">'
    '🎮 TRÌNH MÔ PHỎNG KHO HÀNG FEFO 3D (COLAB READY)'
    '</p></div>'
    + '<iframe src="data:text/html;base64,' + html_b64
    + '" style="width:100%;height:920px;border:2px solid #dc2626;'
    + 'border-radius:18px;box-shadow:0 20px 30px rgba(0,0,0,0.7);"'
    + ' allow="autoplay; fullscreen; pointer-lock"></iframe>'
)
display(HTML(iframe_html))


⏳ Đang tải dữ liệu Kaggle 'willianoliveiragibin/grocery-inventory'...
Using Colab cache for faster access to the 'grocery-inventory' dataset.
✅ Dataset gốc: Grocery_Inventory new v1.csv | 990 bản ghi
📅 Ngày vận hành mô phỏng được tự chọn: 17/05/2024
📦 Nhóm dự kiến từ dữ liệu thật: 🔴 55 gấp, 🟡 80 trung bình, 🟢 627 không gấp
📋 KẾT QUẢ ĐỌC & LỌC DATASET THẬT
  Số dòng gốc: 990 | Hợp lệ ở ngày 17/05/2024: 762
  Khoảng ngày hết hạn đọc được: 25/02/2024 — 24/02/2025
  ⚠️ Loại 228 dòng: đã hết hạn trước ngày mô phỏng
📊 Dữ liệu nguồn hợp lệ theo FEFO (trước khi phân bổ lên kệ):
   GẤP: 55 sản phẩm
   TRUNG BÌNH: 80 sản phẩm
   KHÔNG GẤP: 627 sản phẩm
📦 Sử dụng 762 bản ghi thật từ Kaggle. Ngày vận hành: 17/05/2024.
✅ Đã xuất HTML 3D. Nếu Colab chặn WebGL/iframe, có thể mở file HTML riêng trên Chrome.


In [ ]:
checks = {
    'fleet size': 'const ROBOT_COUNT=5;' in html_code and html_code.count('const ROBOT_COUNT=5;') == 1,
    'five charging bays': html_code.count('{x:1.25,z:') >= 5 and '{x:1.25,z:25.45}' not in html_code,
    'A* route preview': 'function findPathAStar' in html_code and 'findPathDijkstra' not in html_code,
    'continuous swept guard': 'function f5AllowedMove' in html_code,
    'CPA controller': 'function f5PredictiveControl' in html_code,
    'single fleet controller': 'function updateAGVBehavior' not in html_code and 'function dispatchAGV(' not in html_code,
}
assert all(checks.values()), checks
print('Merged notebook checks passed:', checks)
